# lltDomainAdaptation sweep, phase main

Two GPUs, one job list each. Results in `res_*.json`.

In [ ]:
FILES = {'dalab.py': '"""lltDomainAdaptation lab code: data, the evaluation panel, continued pretraining with replay, interpolation,\ntokenizer extension init, RAG prompts and a QLoRA task fine-tune. The notebook cells and the page\'s code blocks\nare excerpts of this file. No backslash in this file: NL = chr(10) stands for a newline.\n"""\nimport json, math, os, random, re, string, time, itertools, collections\nimport torch\nimport torch.nn.functional as F\n\nNL = chr(10)\nMODEL = "Qwen/Qwen3-0.6B-Base"\n\n\n# ---------------------------------------------------------------- data\ndef stream_docs(name, split, n, config=None):\n    from datasets import load_dataset\n    ds = load_dataset(name, name=config, split=split, streaming=True)\n    return [{"id": r["id"], "text": r["text"]} for r in itertools.islice(ds, n)]\n\n\ndef cut(text, n_chars):\n    # The first n_chars characters, cut back to the last space so no word is split.\n    if len(text) <= n_chars:\n        return None\n    return text[:n_chars].rsplit(" ", 1)[0]\n\n\ndef eval_texts(docs, n_chars, n_docs, tok, max_tokens=1023):\n    # Held-out texts: the first n_chars of each long enough doc, kept only if it fits one 1,024-token window.\n    out = []\n    for d in docs:\n        t = cut(d["text"], n_chars)\n        if t and len(tok(t, add_special_tokens=False)["input_ids"]) <= max_tokens:\n            out.append(t)\n        if len(out) == n_docs:\n            break\n    return out\n\n\ndef load_xquad_hi():\n    import pandas as pd\n    from huggingface_hub import hf_hub_download\n    df = pd.read_parquet(hf_hub_download("google/xquad", "xquad.hi/validation-00000-of-00001.parquet", repo_type="dataset"))\n    contexts = list(dict.fromkeys(df["context"]))            # 240 paragraphs, in order of first use\n    cid = {c: i for i, c in enumerate(contexts)}\n    qa = [{"id": r.id, "q": r.question, "a": list(r.answers["text"]), "ctx": cid[r.context]} for r in df.itertuples()]\n    return contexts, qa\n\n\ndef load_xstorycloze(lang):\n    import csv\n    from huggingface_hub import hf_hub_download\n    p = hf_hub_download("juletxara/xstory_cloze", "spring2016.val." + lang + ".tsv.split_20_80_eval.tsv", repo_type="dataset")\n    rows = list(csv.DictReader(open(p, encoding="utf-8"), delimiter=chr(9)))\n    items = []\n    for r in rows:\n        ctx = " ".join(r["input_sentence_" + str(i)] for i in range(1, 5))\n        items.append({"ctx": ctx, "endings": [" " + r["sentence_quiz1"], " " + r["sentence_quiz2"]],\n                      "label": int(r["answer_right_ending"]) - 1})\n    return items\n\n\ndef load_hellaswag(n):\n    import pandas as pd\n    from huggingface_hub import hf_hub_download\n    df = pd.read_parquet(hf_hub_download("Rowan/hellaswag", "data/validation-00000-of-00001.parquet", repo_type="dataset"))\n\n    def prep(t):                                            # the lm-eval-harness preprocessing\n        t = t.strip().replace(" [title]", ". ")\n        t = re.sub(re.escape("[") + ".*?" + re.escape("]"), "", t)\n        return t.replace("  ", " ")\n    items = []\n    for r in df.head(n).itertuples():\n        ctx = prep(r.activity_label + ": " + r.ctx_a + " " + r.ctx_b.capitalize())\n        items.append({"ctx": ctx, "endings": [" " + prep(e) for e in r.endings], "label": int(r.label)})\n    return items\n\n\ndef pack_rows(texts, tok, seq, n_rows):\n    # Concatenate documents, each followed by the end-of-text token, and cut into rows of seq tokens.\n    rows, buf = [], []\n    for t in texts:\n        buf += tok(t, add_special_tokens=False)["input_ids"] + [tok.eos_token_id]\n        while len(buf) >= seq:\n            rows.append(buf[:seq]); buf = buf[seq:]\n            if len(rows) == n_rows:\n                return torch.tensor(rows)\n    raise ValueError("not enough text for " + str(n_rows) + " rows, got " + str(len(rows)))\n\n\ndef mix_rows(hi_rows, en_rows, n_rows, replay, seed):\n    # A replay fraction r: r of the rows come from general English text, the rest from Hindi, in a seeded random order.\n    n_en = round(n_rows * replay)\n    src = [1] * n_en + [0] * (n_rows - n_en)\n    random.Random(seed).shuffle(src)\n    it = [iter(hi_rows), iter(en_rows)]\n    return torch.stack([next(it[s]) for s in src]), n_en\n\n\n# ---------------------------------------------------------------- evaluation panel\n@torch.no_grad()\ndef bits_per_byte(model, tok, texts, bs=2):\n    # Score every token of each text after one end-of-text token. bits per byte = total NLL / ln 2 / total UTF-8 bytes.\n    dev = next(model.parameters()).device\n    nll, ntok = 0.0, 0\n    for i in range(0, len(texts), bs):\n        seqs = [[tok.eos_token_id] + tok(t, add_special_tokens=False)["input_ids"] for t in texts[i:i + bs]]\n        L = max(len(s) for s in seqs)\n        x = torch.full((len(seqs), L), tok.eos_token_id)\n        m = torch.zeros((len(seqs), L), dtype=torch.long)\n        for j, s in enumerate(seqs):\n            x[j, :len(s)] = torch.tensor(s); m[j, :len(s)] = 1\n        x, m = x.to(dev), m.to(dev)\n        with torch.autocast(dev.type, dtype=torch.float16, enabled=dev.type == "cuda"):\n            logits = model(input_ids=x, attention_mask=m).logits\n        lp = torch.log_softmax(logits[:, :-1].float(), -1).gather(-1, x[:, 1:, None])[..., 0]\n        nll -= float((lp * m[:, 1:]).sum()); ntok += int(m[:, 1:].sum())\n    nbytes = sum(len(t.encode("utf-8")) for t in texts)\n    return {"bpb": nll / math.log(2) / nbytes, "nllPerToken": nll / ntok, "tokens": ntok, "bytes": nbytes}\n\n\n@torch.no_grad()\ndef choice_accuracy(model, tok, items, norm_chars=False, bs=16):\n    # Pick the ending with the highest log-likelihood (or per character, the acc_norm of lm-eval-harness).\n    dev = next(model.parameters()).device\n    pairs = [(k, e) for k, it in enumerate(items) for e in it["endings"]]\n    scores = []\n    for i in range(0, len(pairs), bs):\n        chunk = pairs[i:i + bs]\n        seqs, spans = [], []\n        for k, e in chunk:\n            c = tok(items[k]["ctx"], add_special_tokens=False)["input_ids"]\n            ce = tok(items[k]["ctx"] + e, add_special_tokens=False)["input_ids"]\n            n_common = 0\n            while n_common < min(len(c), len(ce)) and c[n_common] == ce[n_common]:\n                n_common += 1\n            seqs.append(ce); spans.append(max(n_common, 1))\n        L = max(len(s) for s in seqs)\n        x = torch.full((len(seqs), L), tok.eos_token_id)\n        m = torch.zeros((len(seqs), L), dtype=torch.long)\n        for j, s in enumerate(seqs):\n            x[j, :len(s)] = torch.tensor(s); m[j, :len(s)] = 1\n        x, m = x.to(dev), m.to(dev)\n        with torch.autocast(dev.type, dtype=torch.float16, enabled=dev.type == "cuda"):\n            logits = model(input_ids=x, attention_mask=m).logits\n        lp = torch.log_softmax(logits[:, :-1].float(), -1).gather(-1, x[:, 1:, None])[..., 0]\n        for j, s in enumerate(seqs):\n            sc = float(lp[j, spans[j] - 1:len(s) - 1].sum())\n            if norm_chars:\n                sc /= len(chunk[j][1])\n            scores.append(sc)\n    correct, p = 0, 0\n    for it in items:\n        n = len(it["endings"])\n        s = scores[p:p + n]; p += n\n        correct += int(max(range(n), key=lambda q: s[q]) == it["label"])\n    return correct / len(items), correct\n\n\ndef panel(model, tok, E, with_tasks=True):\n    # The fixed evaluation panel: Hindi and English held-out bits per byte, plus three small benchmarks.\n    r = {"hiBpb": bits_per_byte(model, tok, E["hi"])["bpb"], "enBpb": bits_per_byte(model, tok, E["en"])["bpb"]}\n    if with_tasks:\n        r["xscHi"], r["xscHiCorrect"] = choice_accuracy(model, tok, E["xsc_hi"])\n        r["xscEn"], r["xscEnCorrect"] = choice_accuracy(model, tok, E["xsc_en"])\n        r["hellaswag"], r["hellaswagCorrect"] = choice_accuracy(model, tok, E["hellaswag"], norm_chars=True)\n    return {k: (round(v, 5) if isinstance(v, float) else v) for k, v in r.items()}\n\n\n# ---------------------------------------------------------------- question answering, closed book and RAG\ndef qa_prompt(q, contexts=(), shots=()):\n    parts = ["प्रश्न: " + sq + NL + "उत्तर: " + sa for sq, sa in shots]\n    parts += ["संदर्भ: " + c for c in contexts]\n    parts.append("प्रश्न: " + q + NL + "उत्तर:")\n    return (NL + NL).join(parts)\n\n\ndef normalize(s):\n    s = "".join(ch for ch in s.lower() if ch not in set(string.punctuation + "।"))\n    return " ".join(s.split())\n\n\ndef f1(pred, gold):\n    p, g = normalize(pred).split(), normalize(gold).split()\n    common = sum((collections.Counter(p) & collections.Counter(g)).values())\n    if common == 0:\n        return 0.0\n    prec, rec = common / len(p), common / len(g)\n    return 2 * prec * rec / (prec + rec)\n\n\n@torch.no_grad()\ndef answer(model, tok, prompts, max_new=40, bs=8):\n    dev = next(model.parameters()).device\n    tok.padding_side = "left"\n    outs = []\n    for i in range(0, len(prompts), bs):\n        enc = tok(prompts[i:i + bs], return_tensors="pt", padding=True, add_special_tokens=False).to(dev)\n        with torch.autocast(dev.type, dtype=torch.float16, enabled=dev.type == "cuda"):\n            g = model.generate(**enc, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.eos_token_id)\n        for row in g[:, enc["input_ids"].shape[1]:]:\n            outs.append(tok.decode(row, skip_special_tokens=True).strip().split(NL)[0].strip())\n    tok.padding_side = "right"\n    return outs\n\n\ndef qa_scores(preds, qa):\n    em = sum(max(float(normalize(p) == normalize(a)) for a in x["a"]) for p, x in zip(preds, qa)) / len(qa)\n    f = sum(max(f1(p, a) for a in x["a"]) for p, x in zip(preds, qa)) / len(qa)\n    return {"em": round(em, 4), "f1": round(f, 4), "n": len(qa)}\n\n\ndef qa_eval(model, tok, Q, modes=("closed0", "closed3", "rag0", "rag3")):\n    # Q holds the test questions, the 3 example question-answer pairs, the paragraph pool and the retrieved ids.\n    out = {}\n    for mode in modes:\n        shots = Q["shots"] if mode.endswith("3") else ()\n        prompts = []\n        for x in Q["test"]:\n            ctx = [Q["pool"][i] for i in Q["retrieved"][x["id"]][:2]] if mode.startswith("rag") else ()\n            prompts.append(qa_prompt(x["q"], ctx, shots))\n        preds = answer(model, tok, prompts)\n        out[mode] = qa_scores(preds, Q["test"])\n        out[mode]["examples"] = [{"q": x["q"], "gold": x["a"][0], "pred": p} for x, p in list(zip(Q["test"], preds))[:5]]\n    return out\n\n\n# ---------------------------------------------------------------- continued pretraining\ndef lr_at(step, steps, peak, warmup, min_ratio=0.1):\n    # Re-warm linearly to the peak, then re-decay with a cosine to min_ratio of the peak.\n    if step < warmup:\n        return peak * (step + 1) / warmup\n    t = (step - warmup) / max(1, steps - warmup)\n    return peak * (min_ratio + (1 - min_ratio) * 0.5 * (1 + math.cos(math.pi * t)))\n\n\ndef train_cpt(model, rows, cfg, on_eval=None):\n    import bitsandbytes as bnb\n    dev = next(model.parameters()).device\n    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n    model.config.use_cache = False\n    model.train()\n    decay = [p for n, p in model.named_parameters() if p.ndim >= 2]\n    no_decay = [p for n, p in model.named_parameters() if p.ndim < 2]\n    opt = bnb.optim.AdamW8bit([{"params": decay, "weight_decay": 0.1}, {"params": no_decay, "weight_decay": 0.0}],\n                              lr=cfg["peak_lr"], betas=(0.9, 0.95))\n    scaler = torch.amp.GradScaler("cuda", init_scale=cfg.get("init_scale", 65536.0))\n    steps, per_step, micro = cfg["steps"], cfg["rows_per_step"], cfg["micro"]\n    warmup = max(1, round(steps * cfg.get("warmup_frac", 0.04)))\n    log, t0 = [], time.time()\n    torch.cuda.reset_peak_memory_stats()\n    for s in range(steps):\n        lr = lr_at(s, steps, cfg["peak_lr"], warmup)\n        for g in opt.param_groups:\n            g["lr"] = lr\n        batch = rows[s * per_step:(s + 1) * per_step]\n        opt.zero_grad(set_to_none=True)\n        tot = 0.0\n        for i in range(0, per_step, micro):\n            x = batch[i:i + micro].to(dev)\n            with torch.autocast("cuda", dtype=torch.float16):\n                loss = model(input_ids=x, labels=x).loss\n            frac = x.shape[0] / per_step\n            tot += float(loss.detach()) * frac\n            scaler.scale(loss * frac).backward()\n        scaler.unscale_(opt)\n        gn = float(torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0))\n        scaler.step(opt); scaler.update()\n        rec = {"step": s + 1, "tokens": (s + 1) * per_step * rows.shape[1], "trainLoss": round(tot, 4), "lr": lr,\n               "gradNorm": round(gn, 3), "scale": scaler.get_scale(), "sec": round(time.time() - t0, 1)}\n        if on_eval and ((s + 1) % cfg["eval_every"] == 0 or s + 1 == steps):\n            opt.zero_grad(set_to_none=True)          # free the gradients before the eval forward passes\n            model.eval(); rec.update(on_eval(model)); model.train()\n        log.append(rec)\n        if (s + 1) % 25 == 0 or s < 3:\n            print(json.dumps(rec), flush=True)\n    model.eval()\n    model.config.use_cache = True                  # generation needs the KV cache back on\n    return log, round(torch.cuda.max_memory_allocated() / 1e9, 2), round(time.time() - t0, 1)\n\n\ndef interpolate(model, base_sd, ft_sd, a):\n    # WiSE-FT: theta(a) = (1 - a) * theta_base + a * theta_ft, parameter by parameter.\n    sd = {k: base_sd[k] + a * (ft_sd[k] - base_sd[k]) for k in ft_sd}\n    model.load_state_dict(sd, strict=False)\n\n\n# ---------------------------------------------------------------- tokenizer extension\ndef add_tokens_mean_init(model, n_vocab, parts):\n    # Grow the (tied) embedding, then set each new row to the mean of the rows of the tokens it was made of.\n    model.resize_token_embeddings(n_vocab, pad_to_multiple_of=64, mean_resizing=False)\n    E = model.get_input_embeddings().weight\n    with torch.no_grad():\n        for new_id, old_ids in parts.items():\n            E[int(new_id)] = E[old_ids].mean(0)\n    return model\n\n\n# ---------------------------------------------------------------- LoRA task fine-tune\ndef qa_train_examples(tok, Q, seed=0):\n    # Prompt = gold paragraph + one other paragraph, in random order, then the question. Loss only on the answer.\n    rng = random.Random(seed)\n    ex = []\n    train_ctx = sorted({x["ctx"] for x in Q["train"]})\n    for x in Q["train"]:\n        other = rng.choice([c for c in train_ctx if c != x["ctx"]])\n        ctx = [Q["contexts"][x["ctx"]], Q["contexts"][other]]\n        rng.shuffle(ctx)\n        p = tok(qa_prompt(x["q"], ctx), add_special_tokens=False)["input_ids"]\n        a = tok(" " + x["a"][0] + NL, add_special_tokens=False)["input_ids"] + [tok.eos_token_id]\n        ex.append((p + a, [-100] * len(p) + a))\n    return ex\n\n\ndef train_lora_qa(qm, tok, ex, epochs=2, bs=8, micro=2, lr=2e-4, seed=0, max_steps=None):\n    # Masked-loss fine-tune of the trainable (LoRA) parameters: batches of bs examples, micro-batches of micro.\n    tr = [p for p in qm.parameters() if p.requires_grad]\n    order = list(range(len(ex))) * epochs\n    random.Random(seed).shuffle(order)\n    steps = len(order) // bs if max_steps is None else max_steps\n    opt = torch.optim.AdamW(tr, lr=lr, weight_decay=0.0)\n    scaler = torch.amp.GradScaler("cuda")\n    qm.train(); qm.config.use_cache = False\n    log, t0 = [], time.time()\n    torch.cuda.reset_peak_memory_stats()\n    for s in range(steps):\n        for g in opt.param_groups:\n            g["lr"] = lr_at(s, steps, lr, max(1, steps // 20), min_ratio=0.0)\n        idx = order[s * bs:(s + 1) * bs]\n        n_ans = sum(sum(1 for v in ex[i][1] if v != -100) for i in idx)\n        opt.zero_grad(set_to_none=True)\n        tot = 0.0\n        for c in range(0, bs, micro):\n            part = idx[c:c + micro]\n            Lm = max(len(ex[i][0]) for i in part)\n            x = torch.full((len(part), Lm), tok.eos_token_id)\n            y = torch.full((len(part), Lm), -100)\n            m = torch.zeros((len(part), Lm), dtype=torch.long)\n            for k, i in enumerate(part):\n                ids, lab = ex[i]\n                x[k, :len(ids)] = torch.tensor(ids); y[k, :len(lab)] = torch.tensor(lab); m[k, :len(ids)] = 1\n            w = sum(sum(1 for v in ex[i][1] if v != -100) for i in part) / n_ans   # weight by answer tokens\n            with torch.autocast("cuda", dtype=torch.float16):\n                loss = qm(input_ids=x.cuda(), attention_mask=m.cuda(), labels=y.cuda()).loss\n            tot += float(loss.detach()) * w\n            scaler.scale(loss * w).backward()\n        scaler.unscale_(opt)\n        torch.nn.utils.clip_grad_norm_(tr, 1.0)\n        scaler.step(opt); scaler.update()\n        log.append({"step": s + 1, "trainLoss": round(tot, 4), "sec": round(time.time() - t0, 1)})\n        if (s + 1) % 30 == 0 or s < 2:\n            print(json.dumps(log[-1]), flush=True)\n    qm.eval(); qm.config.use_cache = True\n    return log, round(torch.cuda.max_memory_allocated() / 1e9, 2), round(time.time() - t0, 1)\n', 'extend_tokenizer.py': '"""Extend Qwen3\'s tokenizer for Hindi by continuing BPE on its own segmentation.\n\nextend(base_id, texts, n_new, mark_regex, out_dir, pin_special=True) -> (out_dir, report).\nUsed by run_extend.py, measure.py and the notebook (the notebook cell is the same code).\n\nmark_regex=True also changes the pre-tokenizer regex so a Devanagari vowel sign (Unicode category M)\nstays inside its word, as Qwen3.5\'s tokenizer does. Every original merge keeps its rank, and every new\nmerge goes after them and is learned from Devanagari pieces only, so text without Devanagari tokenizes exactly as before.\n\npin_special=False reproduces the broken run: the `tokenizers` library numbers added tokens from the size\nof the model vocabulary, so 10,000 new entries push <|endoftext|> from 151643 to 161643, onto a new token.\nThe fix writes each special token into the model vocabulary with its old id.\nNo backslash appears in this file\'s regex edits: B = chr(92) builds them, the same trick the page code uses.\n"""\nimport json, os, collections, hashlib\nfrom tokenizers import Tokenizer, models, trainers, pre_tokenizers\nfrom transformers import AutoTokenizer\n\nB = chr(92)                      # a backslash\nL, M, N = B + "p{L}", B + "p{M}", B + "p{N}"\n\n\ndef mark_regex_edit(regex):\n    # Qwen3: a run of letters may not contain a vowel sign. Qwen3.5: letters and marks together.\n    old1, new1 = "?" + L + "+", "?[" + L + M + "]+"\n    old2, new2 = "[^" + B + "s" + L + N + "]+", "[^" + B + "s" + L + M + N + "]+"\n    assert old1 in regex and old2 in regex, "unexpected pre-tokenizer regex"\n    return regex.replace(old1, new1).replace(old2, new2)\n\n\ndef extend(base_id, texts, n_new, mark_regex, out_dir, pin_special=True):\n    tok = AutoTokenizer.from_pretrained(base_id)\n    j = json.loads(tok.backend_tokenizer.to_str())\n    if mark_regex:\n        split = j["pre_tokenizer"]["pretokenizers"][0]["pattern"]\n        split["Regex"] = mark_regex_edit(split["Regex"])\n    work = Tokenizer.from_str(json.dumps(j))          # original vocab and merges, maybe the new regex\n    vocab = j["model"]["vocab"]                       # byte-level string -> id\n    inv = {i: s for s, i in vocab.items()}\n    # 1. Split the corpus into pieces with the pre-tokenizer and count each Hindi piece.\n    # Only pieces made entirely of Devanagari (U+0900 to U+097F, after one optional leading space) count,\n    # so English, digits, punctuation and code keep their tokens.\n    pieces = collections.Counter()\n    for t in texts:\n        t = work.normalizer.normalize_str(t)\n        for p, (a, b) in work.pre_tokenizer.pre_tokenize_str(t):\n            w = t[a:b].lstrip(" ")\n            if w and all(0x900 <= ord(ch) <= 0x97F for ch in w):\n                pieces[p] += 1\n    # 2. Segment each piece with the ORIGINAL merges, write it as one private-use character per token, run BPE on that.\n    char_of, seg = {}, {}\n    for p in pieces:\n        seg[p] = [x.id for x in work.model.tokenize(p)]\n        for i in seg[p]:\n            if i not in char_of:\n                char_of[i] = chr(0xF0000 + len(char_of))\n    id_of_char = {c: i for i, c in char_of.items()}\n    multi = [(p, c) for p, c in pieces.items() if len(seg[p]) > 1]\n\n    def corpus():\n        for p, c in multi:\n            w = "".join(char_of[i] for i in seg[p])\n            for _ in range(c):\n                yield w\n    bpe = Tokenizer(models.BPE())\n    bpe.pre_tokenizer = pre_tokenizers.WhitespaceSplit()\n    alphabet = list(char_of.values())\n    trainer = trainers.BpeTrainer(vocab_size=len(alphabet) + n_new, min_frequency=2, initial_alphabet=alphabet,\n                                  show_progress=False, limit_alphabet=len(alphabet))\n    if n_new > 0:\n        bpe.train_from_iterator(corpus(), trainer=trainer, length=sum(c for _, c in multi))\n        merges = json.loads(bpe.to_str())["model"]["merges"]\n    else:\n        merges = []\n    to_bytes = lambda s: "".join(inv[id_of_char[ch]] for ch in s)\n    # 3. Append each learned merge after the original ones, with a new id for every new string.\n    first_new = max(a["id"] for a in j["added_tokens"]) + 1     # 151669 for Qwen3\n    if pin_special:\n        for a in j["added_tokens"]:\n            vocab[a["content"]] = a["id"]                        # keeps <|endoftext|> at 151643\n    next_id, new_tokens = first_new, {}\n    for m in merges:\n        a, b = (m.split(" ") if isinstance(m, str) else m)\n        sa, sb = to_bytes(a), to_bytes(b)\n        if sa + sb not in vocab:\n            vocab[sa + sb] = next_id\n            new_tokens[sa + sb] = next_id\n            next_id += 1\n        j["model"]["merges"].append([sa, sb])\n    os.makedirs(out_dir, exist_ok=True)\n    tok.save_pretrained(out_dir)\n    with open(os.path.join(out_dir, "tokenizer.json"), "w", encoding="utf-8") as f:\n        json.dump(j, f, ensure_ascii=False)\n    # 4. For mean-of-subword init: the original token ids each new token is made of.\n    orig = Tokenizer.from_str(tok.backend_tokenizer.to_str()).model\n    parts = {i: [x.id for x in orig.tokenize(s)] for s, i in new_tokens.items()}\n    with open(os.path.join(out_dir, "new_token_parts.json"), "w") as f:\n        json.dump({str(k): v for k, v in parts.items()}, f)\n    h = hashlib.sha256(json.dumps(j["model"]["merges"][len(j["model"]["merges"]) - len(merges):]).encode()).hexdigest()[:16]\n    return out_dir, {"pieces": len(pieces), "alphabet": len(alphabet), "mergesAdded": len(merges),\n                     "newTokens": len(new_tokens), "firstNewId": first_new, "lastNewId": next_id - 1, "mergesSha": h,\n                     "pinSpecial": pin_special, "markRegex": mark_regex}\n', 'worker_da.py': '"""lltDomainAdaptation Kaggle worker: one GPU, a list of jobs, one result file per job.\nUsage: python worker_da.py <jobs.json> <out_dir>   (CUDA_VISIBLE_DEVICES picks the GPU before this starts)\nNeeds dalab.py and extend_tokenizer.py next to it and /kaggle/working/data/*.jsonl from the setup cell."""\nimport gc, json, os, sys, time, traceback, platform\nos.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nimport torch, transformers\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\nimport dalab as L\n\njobs = json.load(open(sys.argv[1])); OUT = sys.argv[2]\nDATA = "/kaggle/working/data"\nrd = lambda n: [json.loads(l) for l in open(DATA + "/" + n + ".jsonl", encoding="utf-8")]\nhi_test, en_edu, hi_train = rd("hi_test"), rd("en_edu"), rd("hi_train")\nsplit = json.load(open(DATA + "/qa_split.json", encoding="utf-8"))\ncontexts, qa = L.load_xquad_hi()\nbyid = {x["id"]: x for x in qa}\ndistract = [t for t in (L.cut(d["text"], 1000) for d in hi_test[200:1000]) if t]\npool = contexts + distract\nimport hashlib\nassert hashlib.sha256("|".join(pool).encode()).hexdigest()[:16] == split["poolSha"], "pool differs from the local index"\nbase_tok = AutoTokenizer.from_pretrained(L.MODEL)\nSMOKE = any(j.get("smoke") for j in jobs)\nnE = (40, 40, 60, 60, 100) if SMOKE else (200, 200, 1511, 1511, 2000)\nE = {"hi": L.eval_texts(hi_test[:400], 1000, nE[0], base_tok), "en": L.eval_texts(en_edu[:500], 3000, nE[1], base_tok),\n     "xsc_hi": L.load_xstorycloze("hi")[:nE[2]], "xsc_en": L.load_xstorycloze("en")[:nE[3]], "hellaswag": L.load_hellaswag(nE[4])}\nEquick = {"hi": E["hi"][:64], "en": E["en"][:64]}\nQ = {"test": [byid[i] for i in split["testIds"]][: (16 if SMOKE else 10 ** 6)], "train": [byid[i] for i in split["trainIds"]],\n     "shots": split["shots"], "pool": pool, "contexts": contexts, "retrieved": split["retrieved"]}\nENV = {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,\n       "transformers": transformers.__version__, "gpu": torch.cuda.get_device_name(0)}\nfor m in ("peft", "bitsandbytes", "datasets", "tokenizers", "accelerate"):\n    try:\n        ENV[m] = __import__(m).__version__\n    except Exception:\n        pass\nprint("eval sizes", {k: len(v) for k, v in E.items()}, "qa test", len(Q["test"]), flush=True)\n\n\ndef load(path, tok_dir=None):\n    tok = AutoTokenizer.from_pretrained(tok_dir or path)\n    model = AutoModelForCausalLM.from_pretrained(path, dtype=torch.float32, device_map="cuda:0")\n    model.eval()\n    return model, tok\n\n\ndef save(res, tag):\n    json.dump(res, open(OUT + "/res_" + tag + ".json", "w", encoding="utf-8"), indent=1, ensure_ascii=False)\n\n\ndef job_panel(j):\n    model, tok = load(j["model"])\n    t = time.time()\n    r = {"panel": L.panel(model, tok, E), "panelSec": round(time.time() - t, 1)}\n    if j.get("qa"):\n        t = time.time(); r["qa"] = L.qa_eval(model, tok, Q); r["qaSec"] = round(time.time() - t, 1)\n    r["params"] = sum(p.numel() for p in model.parameters())\n    return r\n\n\ndef job_cpt(j):\n    model, tok = load(L.MODEL)\n    r = {}\n    if j.get("tokenizer"):\n        from extend_tokenizer import extend\n        t = time.time()\n        tok_dir, rep = extend(L.MODEL, [d["text"] for d in hi_train[:20000]], j["new_tokens"], j["mark_regex"], OUT + "/tok_" + j["tag"])\n        rep["buildSec"] = round(time.time() - t, 1); r["tokenizer"] = rep\n        tok = AutoTokenizer.from_pretrained(tok_dir)\n        parts = json.load(open(tok_dir + "/new_token_parts.json"))\n        if j["init"] == "mean":\n            L.add_tokens_mean_init(model, len(tok), parts)\n        else:\n            model.resize_token_embeddings(len(tok), pad_to_multiple_of=64)    # library default: mean_resizing=True\n        r["vocabRows"] = model.get_input_embeddings().weight.shape[0]\n    r["panel0"] = L.panel(model, tok, E)\n    base_sd = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\n    # training rows: Hindi web docs with the 240 XQuAD paragraphs inserted 4 times each, plus English replay rows\n    import random\n    docs = [d["text"] for d in hi_train[:4000]]\n    rng = random.Random(1234)\n    for c in contexts * 4:\n        docs.insert(rng.randrange(0, 1000), c)\n    n_rows = j["steps"] * j["rows_per_step"]\n    n_en = round(n_rows * j["replay"])\n    hi_rows = L.pack_rows(docs, tok, 1024, n_rows - n_en)\n    en_rows = L.pack_rows([d["text"] for d in en_edu[500:]], tok, 1024, max(n_en, 1))\n    rows, n_en = L.mix_rows(hi_rows, en_rows, n_rows, j["replay"], seed=j.get("seed", 0))\n    r["rows"] = {"total": n_rows, "english": n_en, "tokens": n_rows * 1024}\n    on_eval = lambda m: {"hiBpb": round(L.bits_per_byte(m, tok, Equick["hi"])["bpb"], 5), "enBpb": round(L.bits_per_byte(m, tok, Equick["en"])["bpb"], 5)}\n    cfg = dict(steps=j["steps"], rows_per_step=j["rows_per_step"], micro=j["micro"], peak_lr=j["peak_lr"],\n               eval_every=j["eval_every"], warmup_frac=j.get("warmup_frac", 0.04), init_scale=j.get("init_scale", 65536.0))\n    r["config"] = cfg\n    log, peak, sec = L.train_cpt(model, rows, cfg, on_eval)\n    r.update(metrics=log, peakMemGB=peak, trainSec=sec, tokensPerSec=round(n_rows * 1024 / sec, 1))\n    r["panelFinal"] = L.panel(model, tok, E)\n    if j.get("qa"):\n        r["qa"] = L.qa_eval(model, tok, Q)\n    ft_sd = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\n    if j.get("save"):\n        model.half().save_pretrained(OUT + "/ckpt_" + j["tag"], safe_serialization=True)\n        tok.save_pretrained(OUT + "/ckpt_" + j["tag"])\n        model.float()\n    r["interp"] = []\n    for a in j.get("interp", []):\n        L.interpolate(model, base_sd, ft_sd, a)\n        r["interp"].append({"a": a, **L.panel(model, tok, E)})\n        print("interp", r["interp"][-1], flush=True)\n    return r\n\n\ndef job_extinit(j):\n    # Step-0 effect of the tokenizer extension with the library\'s default resize (mean of all rows), no training.\n    from extend_tokenizer import extend\n    model, tok = load(L.MODEL)\n    tok_dir, rep = extend(L.MODEL, [d["text"] for d in hi_train[:20000]], j["new_tokens"], j["mark_regex"], OUT + "/tok_" + j["tag"])\n    tok = AutoTokenizer.from_pretrained(tok_dir)\n    model.resize_token_embeddings(len(tok), pad_to_multiple_of=64)\n    return {"tokenizer": rep, "vocabRows": model.get_input_embeddings().weight.shape[0], "panel0": L.panel(model, tok, E)}\n\n\ndef job_qlora(j):\n    from transformers import BitsAndBytesConfig\n    from peft import LoraConfig, get_peft_model\n    q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,\n                           bnb_4bit_compute_dtype=torch.float16)\n    src = j["model"]\n    tok = AutoTokenizer.from_pretrained(src)\n    model = AutoModelForCausalLM.from_pretrained(src, quantization_config=q, dtype=torch.float16, device_map="cuda:0")\n    for p in model.parameters():\n        p.requires_grad = False\n    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n    model.enable_input_require_grads()\n    model.config.use_cache = False\n    model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, target_modules="all-linear", task_type="CAUSAL_LM"))\n    for p in model.parameters():\n        if p.requires_grad and p.dtype != torch.float32:\n            p.data = p.data.float()\n    tr = [p for p in model.parameters() if p.requires_grad]\n    r = {"trainable": sum(p.numel() for p in tr), "total": sum(p.numel() for p in model.parameters())}\n    model.eval(); model.config.use_cache = True\n    with model.disable_adapter():\n        r["panelBase4bit"] = L.panel(model, tok, E)\n        r["qaBase4bit"] = L.qa_eval(model, tok, Q, modes=("closed0", "rag0", "rag3"))\n    ex = L.qa_train_examples(tok, Q, seed=0)\n    bs = j["batch"]\n    log, peak, sec = L.train_lora_qa(model, tok, ex, epochs=j["epochs"], bs=bs, micro=j.get("micro", 2), lr=j["lr"],\n                                     max_steps=6 if j.get("smoke") else None)\n    steps = len(log)\n    r.update(metrics=log, steps=steps, trainSec=sec, peakMemGB=peak, trainExamples=len(ex), epochs=j["epochs"], batch=bs, lr=j["lr"])\n    r["panel"] = L.panel(model, tok, E)\n    r["qa"] = L.qa_eval(model, tok, Q, modes=("closed0", "rag0", "rag3"))\n    return r\n\n\nfor j in jobs:\n    t = time.time()\n    res = {"tag": j["tag"], "job": j, "env": ENV, "date": time.strftime("%Y-%m-%d"),\n           "evalSizes": {k: len(v) for k, v in E.items()}, "qaTest": len(Q["test"])}\n    try:\n        res.update({"panel": job_panel, "cpt": job_cpt, "qlora": job_qlora, "extinit": job_extinit}[j["kind"]](j)); res["ok"] = True\n    except Exception as e:\n        res["ok"] = False; res["error"] = type(e).__name__ + ": " + str(e)[:1500]; res["trace"] = traceback.format_exc()[-3000:]\n        print(res["trace"], flush=True)\n    res["wallSec"] = round(time.time() - t, 1)\n    save(res, j["tag"])\n    print("JOB DONE", j["tag"], res.get("ok"), res["wallSec"], flush=True)\n    gc.collect(); torch.cuda.empty_cache()\n'}
SPLIT = '{"poolSha": "9759f51894d45de6", "testIds": ["5727213c708984140094da35", "5727213c708984140094da36", "5727213c708984140094da38", "5727213c708984140094da39", "5727213c708984140094da37", "5728d63c4b864d1900164f18", "5728d63c4b864d1900164f19", "5728d63c4b864d1900164f1a", "5728d63c4b864d1900164f1c", "5728d63c4b864d1900164f1b", "5728202c4b864d19001644ec", "5728202c4b864d19001644ed", "5728202c4b864d19001644ee", "5728202c4b864d19001644ef", "5728202c4b864d19001644f0", "5728e07e3acd2414000e00e9", "5728e07e3acd2414000e00ea", "5728e07e3acd2414000e00eb", "5728e07e3acd2414000e00ec", "5728e07e3acd2414000e00ed", "57282dfb4b864d1900164668", "57282dfb4b864d1900164669", "57282dfb4b864d190016466a", "57282dfb4b864d190016466b", "57282dfb4b864d190016466c", "5728e715ff5b5019007da914", "5728e715ff5b5019007da916", "5728e715ff5b5019007da917", "5728e715ff5b5019007da918", "5728eb1a3acd2414000e01c5", "5728eb1a3acd2414000e01c6", "5728eb1a3acd2414000e01c7", "5728eb1a3acd2414000e01c8", "5728eb1a3acd2414000e01c9", "5728eef92ca10214002daab0", "5728eef92ca10214002daab1", "5728eef92ca10214002daab2", "5728eef92ca10214002daab3", "5728eef92ca10214002daab4", "57273a465951b619008f86ff", "57273a465951b619008f8700", "57273a465951b619008f8701", "57273a465951b619008f8702", "57273a465951b619008f8703", "57273e50dd62a815002e9a02", "57273e50dd62a815002e9a03", "57273e50dd62a815002e9a04", "57273e50dd62a815002e9a05", "57273f27dd62a815002e9a0a", "57273f27dd62a815002e9a0b", "57273f27dd62a815002e9a0c", "57273f27dd62a815002e9a0d", "57274e0d708984140094dbe5", "57274e0d708984140094dbe6", "57274e0d708984140094dbe7", "57274e0d708984140094dbe8", "572750e8dd62a815002e9af0", "572750e8dd62a815002e9af1", "572750e8dd62a815002e9af2", "572750e8dd62a815002e9af3", "572750e8dd62a815002e9af4", "57274b35f1498d1400e8f5d4", "57274b35f1498d1400e8f5d5", "57274b35f1498d1400e8f5d6", "57274b35f1498d1400e8f5d7", "57274beff1498d1400e8f5e4", "57274beff1498d1400e8f5e5", "57274beff1498d1400e8f5e6", "57274beff1498d1400e8f5e7", "5727515f708984140094dc11", "5727515f708984140094dc12", "5727515f708984140094dc13", "5727515f708984140094dc14", "5727515f708984140094dc15", "57275409708984140094dc35", "57275409708984140094dc36", "57275409708984140094dc37", "572757bef1498d1400e8f690", "572757bef1498d1400e8f691", "572757bef1498d1400e8f692", "572757bef1498d1400e8f693", "572757bef1498d1400e8f694", "5727aec03acd2414000de991", "5727aec03acd2414000de992", "5727aec03acd2414000de993", "5727aec03acd2414000de994", "5727aec03acd2414000de995", "5727cc15ff5b5019007d9576", "5727cc15ff5b5019007d9577", "5727cc15ff5b5019007d9578", "5727d0f73acd2414000ded13", "5727d0f73acd2414000ded14", "5727d0f73acd2414000ded15", "5727d1c93acd2414000ded3f", "5727d1c93acd2414000ded40", "5727d1c93acd2414000ded41", "5727d1c93acd2414000ded42", "5727de862ca10214002d9860", "5727de862ca10214002d9861", "5727de862ca10214002d9862", "5727de862ca10214002d9863", "5727c94bff5b5019007d954a", "5727c94bff5b5019007d954b", "5727c94bff5b5019007d954c", "5727c94bff5b5019007d954d", "5727c94bff5b5019007d954e", "5727cb4b2ca10214002d9676", "5727cb4b2ca10214002d9677", "5727cb4b2ca10214002d9678", "5727cb4b2ca10214002d9679", "5727cb4b2ca10214002d967a", "57280fd3ff5b5019007d9c26", "57280fd3ff5b5019007d9c27", "57280fd3ff5b5019007d9c28", "57280fd3ff5b5019007d9c29", "57280fd3ff5b5019007d9c2a", "57281ab63acd2414000df493", "57281ab63acd2414000df494", "57281ab63acd2414000df495", "57281ab63acd2414000df496", "57281ab63acd2414000df497", "572824f13acd2414000df58f", "572824f13acd2414000df590", "572824f13acd2414000df591", "572824f13acd2414000df592", "572824f13acd2414000df593", "572a1046af94a219006aa78d", "572a1046af94a219006aa78e", "572a1046af94a219006aa78f", "572a1046af94a219006aa790", "572a1046af94a219006aa791", "572a11663f37b31900478693", "572a11663f37b31900478694", "572a11663f37b31900478695", "572a11663f37b31900478696", "572a11663f37b31900478697", "572a13841d0469140077973b", "572a13841d0469140077973c", "572a13841d0469140077973d", "572a13841d0469140077973e", "572a13841d0469140077973f", "572a18a4af94a219006aa7df", "572a18a4af94a219006aa7e0", "572a18a4af94a219006aa7e1", "572a18a4af94a219006aa7e2", "572a18a4af94a219006aa7e3", "572a1a5c6aef051400155284", "572a1a5c6aef051400155285", "572a1a5c6aef051400155286", "572a1a5c6aef051400155287", "572a1a5c6aef051400155288", "5727f3193acd2414000df0a5", "5727f3193acd2414000df0a6", "5727f3193acd2414000df0a7", "5727f3193acd2414000df0a8", "5727f3193acd2414000df0a9", "57280f0d3acd2414000df35b", "57280f0d3acd2414000df35c", "57280f0d3acd2414000df35d", "57280f0d3acd2414000df35e", "57280f0d3acd2414000df35f", "572811434b864d190016438c", "572811434b864d190016438d", "572811434b864d190016438e", "572811434b864d190016438f", "572811434b864d1900164390", "57281a952ca10214002d9dea", "57281a952ca10214002d9deb", "57281a952ca10214002d9dec", "57281a952ca10214002d9ded", "57281a952ca10214002d9dee", "572828383acd2414000df5c3", "572828383acd2414000df5c4", "572828383acd2414000df5c5", "572828383acd2414000df5c6", "572828383acd2414000df5c7", "57284b904b864d19001648e2", "57284b904b864d19001648e3", "57284b904b864d19001648e4", "57284b904b864d19001648e5", "57284b904b864d19001648e6", "57284d484b864d1900164900", "57284d484b864d1900164901", "57284d484b864d1900164902", "57284d484b864d1900164903", "57284d484b864d1900164904", "572855973acd2414000df925", "572855973acd2414000df926", "572855973acd2414000df927", "572855973acd2414000df928", "572855973acd2414000df929", "57286951ff5b5019007da20e", "57286951ff5b5019007da20f", "57286951ff5b5019007da210", "57286951ff5b5019007da211", "57286951ff5b5019007da212", "572870b2ff5b5019007da222", "572870b2ff5b5019007da223", "572870b2ff5b5019007da224", "572870b2ff5b5019007da225", "57286dfa2ca10214002da332", "57286dfa2ca10214002da333", "57286dfa2ca10214002da334", "57286dfa2ca10214002da335", "57286dfa2ca10214002da336", "57286fa83acd2414000df9e5", "57286fa83acd2414000df9e6", "57286fa83acd2414000df9e7", "57286fa83acd2414000df9e8", "57286fa83acd2414000df9e9", "57287d4a2ca10214002da3e4", "57287d4a2ca10214002da3e5", "57287d4a2ca10214002da3e6", "57287d4a2ca10214002da3e7", "57287d4a2ca10214002da3e8", "572881d34b864d1900164a5a", "572881d34b864d1900164a5b", "572881d34b864d1900164a5c", "572881d34b864d1900164a5d", "572881d34b864d1900164a5e", "5728848cff5b5019007da298", "5728848cff5b5019007da299", "5728848cff5b5019007da29a", "5728848cff5b5019007da29b", "57290b21af94a219006a9fcf", "57290b21af94a219006a9fd0", "57290b21af94a219006a9fd1", "57290b21af94a219006a9fd2", "57290ee2af94a219006a9fff", "57290ee2af94a219006aa000", "57290ee2af94a219006aa001", "57290ee2af94a219006aa002", "57290ee2af94a219006aa003", "572914f46aef051400154a46", "572914f46aef051400154a47", "572914f46aef051400154a48", "572914f46aef051400154a49", "572920d73f37b31900478055", "572920d73f37b31900478056", "572920d73f37b31900478057", "572920d73f37b31900478058", "5729281baf94a219006aa11f", "5729281baf94a219006aa120", "5729281baf94a219006aa121", "5729281baf94a219006aa122", "57293bc91d0469140077919b", "57293bc91d0469140077919c", "57293bc91d0469140077919d", "57293bc91d0469140077919e", "57293bc91d0469140077919f", "57293e221d046914007791d5", "57293e221d046914007791d6", "57293e221d046914007791d7", "57293e221d046914007791d8", "57293e221d046914007791d9", "572940973f37b319004781a5", "572940973f37b319004781a6", "572940973f37b319004781a7", "572940973f37b319004781a8", "57294209af94a219006aa201", "57294209af94a219006aa202", "57294209af94a219006aa203", "57294209af94a219006aa204", "57294e6b1d04691400779275", "57294e6b1d04691400779276", "57294e6b1d04691400779277", "572957ad1d046914007792d9", "572957ad1d046914007792da", "572957ad1d046914007792db", "572957ad1d046914007792dc", "572957ad1d046914007792dd", "5729582b1d046914007792e3", "5729582b1d046914007792e4", "5729582b1d046914007792e5", "5729582b1d046914007792e6", "5729582b1d046914007792e7", "572968cf1d046914007793cb", "572968cf1d046914007793cc", "572968cf1d046914007793cd", "572968cf1d046914007793ce", "572968cf1d046914007793cf", "57296b151d046914007793f1", "57296b151d046914007793f2", "57296b151d046914007793f3", "57296b151d046914007793f4", "57296d1b1d0469140077940d", "57296d1b1d0469140077940e", "57296d1b1d0469140077940f", "57296d571d04691400779413", "57296d571d04691400779414", "57296d571d04691400779415", "57296d571d04691400779416", "57296d571d04691400779417", "57296f293f37b319004783a3", "57296f293f37b319004783a4", "57296f293f37b319004783a5", "57296f293f37b319004783a6", "57296f293f37b319004783a7", "572991943f37b319004784a1", "572991943f37b319004784a2", "572991943f37b319004784a3", "572991943f37b319004784a4", "572991943f37b319004784a5", "57299a6f6aef051400155016", "57299a6f6aef051400155017", "57299a6f6aef051400155019", "57299a6f6aef05140015501a", "57299a6f6aef051400155018", "5729a26d6aef05140015505a", "5729a26d6aef05140015505b", "5729a26d6aef05140015505c", "5729a26d6aef05140015505d", "572f6a0ba23a5019007fc5eb", "572f6a0ba23a5019007fc5ec", "572f6a0ba23a5019007fc5ed", "572f6a0ba23a5019007fc5ee", "572ff12e04bcaa1900d76eff", "572ff12e04bcaa1900d76f00", "572ff12e04bcaa1900d76f02", "572ff12e04bcaa1900d76f03", "572ff673b2c2fd1400568669", "572ff673b2c2fd140056866a", "572ff673b2c2fd140056866b", "572ffd75b2c2fd14005686e5", "572ffd75b2c2fd14005686e6", "572ffd75b2c2fd14005686e7", "572ffd75b2c2fd14005686e8", "572ffd75b2c2fd14005686e9", "572fffb404bcaa1900d76fef", "572fffb404bcaa1900d76ff0", "572fffb404bcaa1900d76ff1", "572fffb404bcaa1900d76ff2", "572fffb404bcaa1900d76ff3", "57300a9a04bcaa1900d77063", "57300a9a04bcaa1900d77064", "57300a9a04bcaa1900d77065", "57300a9a04bcaa1900d77066", "57300a9a04bcaa1900d77067", "572fadcbb2c2fd1400568329", "572fadcbb2c2fd140056832a", "572fadcbb2c2fd140056832c", "572fadcbb2c2fd140056832b", "572fadcbb2c2fd140056832d", "572fc6f204bcaa1900d76cf5", "572fc6f204bcaa1900d76cf6", "572fc6f204bcaa1900d76cf7", "572fc6f204bcaa1900d76cf8", "572fc6f204bcaa1900d76cf9", "572fc8a904bcaa1900d76d1f", "572fc8a904bcaa1900d76d20", "572fc8a904bcaa1900d76d21", "572fc8a904bcaa1900d76d22", "572fcc43b2c2fd140056847d", "572fcc43b2c2fd140056847e", "572fcc43b2c2fd140056847f", "572fcc43b2c2fd1400568480", "572fdb17b2c2fd140056851d", "572fdb17b2c2fd140056851e", "572fdb17b2c2fd140056851f", "572fdb17b2c2fd1400568520", "572fdb17b2c2fd1400568521", "572ff932a23a5019007fcbd5", "572ff932a23a5019007fcbd6", "572ff932a23a5019007fcbd7", "572ff932a23a5019007fcbd8", "572ff932a23a5019007fcbd9", "572ffe6fb2c2fd14005686ef", "572ffe6fb2c2fd14005686f2", "572ffe6fb2c2fd14005686f0", "572ffe6fb2c2fd14005686f3", "572ffe6fb2c2fd14005686f1", "572ffee1947a6a140053cf14", "572ffee1947a6a140053cf15", "572ffee1947a6a140053cf17", "572ffee1947a6a140053cf16", "572ffee1947a6a140053cf18", "57302700a23a5019007fce89", "57302700a23a5019007fce8a", "57302700a23a5019007fce8b", "57302700a23a5019007fce8c", "57302700a23a5019007fce8d", "57302bd0b2c2fd14005689db", "57302bd0b2c2fd14005689dc", "57302bd0b2c2fd14005689de", "57302bd0b2c2fd14005689df", "57302bd0b2c2fd14005689dd", "573088da069b53140083216b", "573088da069b53140083216c", "573088da069b53140083216d", "573088da069b53140083216e", "573088da069b53140083216f", "57308f6b8ab72b1400f9c580", "57308f6b8ab72b1400f9c581", "57308f6b8ab72b1400f9c582", "57308f6b8ab72b1400f9c583", "57308f6b8ab72b1400f9c584", "573093598ab72b1400f9c5ae", "573093598ab72b1400f9c5af", "573093598ab72b1400f9c5b0", "573093598ab72b1400f9c5b1", "573093598ab72b1400f9c5b2", "57309ef18ab72b1400f9c600", "57309ef18ab72b1400f9c602", "57309ef18ab72b1400f9c601", "57309ef18ab72b1400f9c603", "57309ef18ab72b1400f9c604", "5730a4d02461fd1900a9cf29", "5730a4d02461fd1900a9cf2a", "5730a4d02461fd1900a9cf2d", "5730a4d02461fd1900a9cf2b", "5730a4d02461fd1900a9cf2c", "57309921396df919000961f6", "57309921396df919000961f7", "57309921396df919000961f8", "5730b2312461fd1900a9cfad", "5730b2312461fd1900a9cfae", "5730b2312461fd1900a9cfaf", "5730b2ac2461fd1900a9cfb3", "5730b2ac2461fd1900a9cfb6", "5730b2ac2461fd1900a9cfb4", "5730b2ac2461fd1900a9cfb5", "5730b9852461fd1900a9cff9", "5730b9852461fd1900a9cffa", "5730b9852461fd1900a9cffb", "5730d26fb54a4f140068cc94", "5730d26fb54a4f140068cc95", "5730d26fb54a4f140068cc96", "5730d26fb54a4f140068cc97", "5733d4c8d058e614000b6353", "5733d4c8d058e614000b6354", "5733d4c8d058e614000b6355", "5733d4c8d058e614000b6356", "5733d68ed058e614000b637f", "5733d68ed058e614000b6380", "5733d68ed058e614000b6381", "5733f1784776f41900661575", "5733f1784776f41900661576", "5733f1784776f41900661577", "5733f309d058e614000b6648", "5733f309d058e614000b6649", "5733f309d058e614000b664a", "5733fb7bd058e614000b66ff", "5733fb7bd058e614000b6700", "5733fb7bd058e614000b6701", "57373d0cc3c5551400e51e85", "57373d0cc3c5551400e51e86", "57373d0cc3c5551400e51e87", "57373d0cc3c5551400e51e88", "57373d0cc3c5551400e51e89", "5737821cc3c5551400e51f19", "5737821cc3c5551400e51f1a", "5737821cc3c5551400e51f1b", "5737821cc3c5551400e51f1c", "57378b141c4567190057449f", "57378b141c456719005744a0", "57378b141c456719005744a1", "57378b141c456719005744a2", "57378c9b1c456719005744a8", "57378c9b1c456719005744a9", "57378c9b1c456719005744a7", "57378c9b1c456719005744aa", "5737a25ac3c5551400e51f51", "5737a25ac3c5551400e51f52", "5737a25ac3c5551400e51f53", "5737a25ac3c5551400e51f54"], "trainIds": ["56beb4343aeaaa14008c925b", "56beb4343aeaaa14008c925c", "56beb4343aeaaa14008c925d", "56beb4343aeaaa14008c925e", "56beb4343aeaaa14008c925f", "56d6f3500d65d21400198290", "56d6f3500d65d21400198291", "56d6f3500d65d21400198292", "56d6f3500d65d21400198293", "56d6f3500d65d21400198294", "56d9992fdc89441400fdb59c", "56d9992fdc89441400fdb59e", "56d9992fdc89441400fdb59f", "56d9992fdc89441400fdb5a0", "56beb7953aeaaa14008c92ab", "56beb7953aeaaa14008c92ac", "56beb7953aeaaa14008c92ad", "56beb7953aeaaa14008c92ae", "56beb7953aeaaa14008c92af", "56bf36b93aeaaa14008c9561", "56bf36b93aeaaa14008c9562", "56bf36b93aeaaa14008c9563", "56bf36b93aeaaa14008c9564", "56bf36b93aeaaa14008c9565", "56d7018a0d65d214001982c2", "56d7018a0d65d214001982c3", "56d7018a0d65d214001982c5", "56d99f99dc89441400fdb628", "56d99f99dc89441400fdb629", "56d99f99dc89441400fdb62c", "56beb86b3aeaaa14008c92bd", "56beb86b3aeaaa14008c92be", "56beb86b3aeaaa14008c92bf", "56beb86b3aeaaa14008c92c0", "56beb86b3aeaaa14008c92c1", "56bf3fd53aeaaa14008c9591", "56bf3fd53aeaaa14008c9592", "56bf3fd53aeaaa14008c9593", "56bf3fd53aeaaa14008c9594", "56bf3fd53aeaaa14008c9595", "56d704430d65d214001982de", "56d704430d65d214001982e0", "56d704430d65d214001982e1", "56d704430d65d214001982e2", "56d9a0eadc89441400fdb63e", "56d9a0eadc89441400fdb63f", "56d9a0eadc89441400fdb640", "56bec6ac3aeaaa14008c93fd", "56bec6ac3aeaaa14008c93fe", "56bec6ac3aeaaa14008c93ff", "56bec6ac3aeaaa14008c9400", "56bec6ac3aeaaa14008c9401", "56d20650e7d4791d00902614", "56d20650e7d4791d00902615", "56d7253b0d65d214001983d4", "56d7253b0d65d214001983d5", "56d9c455dc89441400fdb7c4", "56d9c455dc89441400fdb7c5", "56d9c455dc89441400fdb7c6", "56beca913aeaaa14008c946d", "56beca913aeaaa14008c946e", "56beca913aeaaa14008c946f", "56beca913aeaaa14008c9470", "56beca913aeaaa14008c9471", "56d726b60d65d214001983ea", "56d726b60d65d214001983eb", "56d726b60d65d214001983ec", "56d726b60d65d214001983ed", "56d726b60d65d214001983ee", "56d9cb47dc89441400fdb832", "56d9cb47dc89441400fdb833", "56d9cb47dc89441400fdb834", "56d9cb47dc89441400fdb835", "56d9cb47dc89441400fdb836", "57339c16d058e614000b5ec5", "57339c16d058e614000b5ec6", "57339c16d058e614000b5ec7", "57339c16d058e614000b5ec8", "57339c16d058e614000b5ec9", "5733a32bd058e614000b5f32", "5733a32bd058e614000b5f33", "5733a32bd058e614000b5f34", "5733a32bd058e614000b5f35", "5733a32bd058e614000b5f36", "57338007d058e614000b5bda", "57338007d058e614000b5bdb", "57338007d058e614000b5bdc", "57338007d058e614000b5bdd", "57338007d058e614000b5bde", "573380e0d058e614000b5be9", "573380e0d058e614000b5bea", "573380e0d058e614000b5beb", "5733834ed058e614000b5c26", "5733834ed058e614000b5c27", "5733834ed058e614000b5c28", "5733834ed058e614000b5c29", "5733834ed058e614000b5c2a", "56dde1d966d3e219004dad8d", "56de0daecffd8e1900b4b595", "56de0daecffd8e1900b4b596", "56de0f6a4396321400ee257f", "56de10b44396321400ee2593", "56de10b44396321400ee2594", "56de10b44396321400ee2595", "56de49564396321400ee277a", "56dfa0d84a1a83140091ebb7", "56dfa0d84a1a83140091ebb8", "56dfa0d84a1a83140091ebb9", "56dfa0d84a1a83140091ebba", "56e0bb9f7aa994140058e6cb", "56e0bb9f7aa994140058e6cc", "56e0bb9f7aa994140058e6cd", "56e0bb9f7aa994140058e6ce", "56e0bb9f7aa994140058e6cf", "56dfb5777aa994140058e021", "56dfb5777aa994140058e022", "56dfb5777aa994140058e023", "56dfb5777aa994140058e024", "56dfb5777aa994140058e025", "56e0d6cf231d4119001ac420", "56e0d6cf231d4119001ac421", "56e0d6cf231d4119001ac422", "56e0d6cf231d4119001ac423", "56e0d6cf231d4119001ac424", "56e0fc3f7aa994140058e877", "56e0fc3f7aa994140058e878", "56e0fc3f7aa994140058e879", "56e0fc3f7aa994140058e87a", "56e0fc3f7aa994140058e87b", "56e10aa5cd28a01900c674b3", "56e10aa5cd28a01900c674b4", "56e10aa5cd28a01900c674b5", "56e1254ae3433e1400422c66", "56e1254ae3433e1400422c67", "56e1254ae3433e1400422c68", "56e16182e3433e1400422e28", "56e16182e3433e1400422e29", "56e16182e3433e1400422e2a", "56e181d9e3433e1400422fa0", "56e181d9e3433e1400422fa1", "56e181d9e3433e1400422fa2", "56e181d9e3433e1400422fa3", "56e181d9e3433e1400422fa4", "56e1a0dccd28a01900c67a2e", "56e1a0dccd28a01900c67a2f", "56e1b62ecd28a01900c67aa3", "56e1b62ecd28a01900c67aa4", "56e1b62ecd28a01900c67aa5", "56e1b62ecd28a01900c67aa6", "56e1ee4de3433e1400423210", "56e1ee4de3433e1400423211", "56e1ee4de3433e1400423212", "56e1ee4de3433e1400423214", "56e7586d37bdd419002c3eb3", "56e7586d37bdd419002c3eb4", "56e7586d37bdd419002c3eb5", "56e7586d37bdd419002c3eb6", "56e7586d37bdd419002c3eb7", "56e7788200c9c71400d77180", "56e7788200c9c71400d77181", "56e7788200c9c71400d77182", "56e7788200c9c71400d77183", "56e7788200c9c71400d77184", "56e7796637bdd419002c3ffd", "56e7796637bdd419002c3ffe", "56e7796637bdd419002c3fff", "56e7796637bdd419002c4000", "56e7796637bdd419002c4001", "56e77cee00c9c71400d771a8", "56e77cee00c9c71400d771a9", "56e77cee00c9c71400d771aa", "56e77cee00c9c71400d771ab", "56e77da237bdd419002c403b", "56e77da237bdd419002c403c", "56e77da237bdd419002c403d", "56e77da237bdd419002c403e", "56e77da237bdd419002c403f", "56f8094aa6d7ea1400e17391", "56f8094aa6d7ea1400e17392", "56f8094aa6d7ea1400e17393", "56f8094aa6d7ea1400e17394", "56f8094aa6d7ea1400e17395", "56f84485aef2371900625f71", "56f84485aef2371900625f72", "56f84485aef2371900625f73", "56f84485aef2371900625f74", "56f84485aef2371900625f75", "56f86e91aef2371900626067", "56f86e91aef2371900626068", "56f86e91aef2371900626069", "56f86e91aef237190062606a", "56f86e91aef237190062606b", "56f8720eaef237190062608f", "56f8720eaef2371900626090", "56f8720eaef2371900626091", "56f8720eaef2371900626092", "56f8720eaef2371900626093", "56f8ca289b226e1400dd1007", "56f8ca289b226e1400dd1008", "56f8ca289b226e1400dd1009", "5705f7c875f01819005e77dc", "5705f7c875f01819005e77dd", "5705f7c875f01819005e77de", "5705f7c875f01819005e77df", "5705f7c875f01819005e77e0", "5706074552bb8914006897d4", "5706074552bb8914006897d5", "5706074552bb8914006897d6", "5706074552bb8914006897d7", "5706074552bb8914006897d8", "570610b275f01819005e792a", "570610b275f01819005e792b", "570610b275f01819005e792c", "570610b275f01819005e792d", "570610b275f01819005e792e", "5706143575f01819005e7950", "5706143575f01819005e7951", "5706143575f01819005e7952", "5706143575f01819005e7953", "5706143575f01819005e7954", "5706149552bb891400689880", "5706149552bb891400689881", "5706149552bb891400689882", "5706149552bb891400689883", "5706149552bb891400689884", "570966e0200fba1400367f4f", "570966e0200fba1400367f50", "570966e0200fba1400367f51", "570966e0200fba1400367f52", "570966e0200fba1400367f53", "570967c4ed30961900e840ba", "570967c4ed30961900e840bb", "570967c4ed30961900e840bc", "570967c4ed30961900e840bd", "570967c4ed30961900e840be", "57096b66200fba1400367fa7", "57096b66200fba1400367fa8", "57096b66200fba1400367fa9", "57096b66200fba1400367faa", "57096b66200fba1400367fab", "57097c8fed30961900e841f2", "57097c8fed30961900e841f3", "57097c8fed30961900e841f4", "57097c8fed30961900e841f5", "57097c8fed30961900e841f6", "57097d63ed30961900e841fc", "57097d63ed30961900e841fd", "57097d63ed30961900e841fe", "57097d63ed30961900e841ff", "57097d63ed30961900e84200", "570d28bdb3d812140066d4a3", "570d28bdb3d812140066d4a4", "570d28bdb3d812140066d4a5", "570d28bdb3d812140066d4a6", "570d28bdb3d812140066d4a7", "570d2f5bfed7b91900d45ccf", "570d2f5bfed7b91900d45cd0", "570d2f5bfed7b91900d45cd1", "570d2f5bfed7b91900d45cd2", "570d2f5bfed7b91900d45cd3", "570d3468b3d812140066d543", "570d3468b3d812140066d544", "570d3468b3d812140066d545", "570d3468b3d812140066d546", "570d3468b3d812140066d547", "570d47b8b3d812140066d62d", "570d47b8b3d812140066d62e", "570d47b8b3d812140066d62f", "570d47b8b3d812140066d630", "570d47b8b3d812140066d631", "570d4a6bfed7b91900d45e13", "570d4a6bfed7b91900d45e14", "570d4a6bfed7b91900d45e15", "570d4a6bfed7b91900d45e16", "57107d73b654c5140001f91d", "57107d73b654c5140001f91e", "57107d73b654c5140001f91f", "57107d73b654c5140001f920", "5710eca0a58dae1900cd6b3a", "5710eca0a58dae1900cd6b3b", "5710eca0a58dae1900cd6b3c", "5710eca0a58dae1900cd6b3d", "5710eca0a58dae1900cd6b3e", "57111380a58dae1900cd6bd6", "57111380a58dae1900cd6bd7", "57111380a58dae1900cd6bd8", "57111380a58dae1900cd6bd9", "57111380a58dae1900cd6bda", "57111713a58dae1900cd6c00", "57111713a58dae1900cd6c01", "57111713a58dae1900cd6c02", "57111713a58dae1900cd6c03", "57111713a58dae1900cd6c04", "57111b95a58dae1900cd6c50", "57111b95a58dae1900cd6c51", "57111b95a58dae1900cd6c52", "57111b95a58dae1900cd6c53", "57111b95a58dae1900cd6c54", "571144d1a58dae1900cd6d6e", "571144d1a58dae1900cd6d6f", "571144d1a58dae1900cd6d70", "571144d1a58dae1900cd6d71", "57114667a58dae1900cd6d80", "57114667a58dae1900cd6d81", "57114667a58dae1900cd6d82", "57114667a58dae1900cd6d83", "57114667a58dae1900cd6d84", "57114e8d50c2381900b54a5b", "57114e8d50c2381900b54a5c", "57114e8d50c2381900b54a5d", "57114e8d50c2381900b54a5f", "57114e8d50c2381900b54a5e", "57115bf350c2381900b54a93", "57115bf350c2381900b54a94", "57115bf350c2381900b54a95", "57115bf350c2381900b54a96", "57115bf350c2381900b54a97", "57115ff82419e314009555c3", "57115ff82419e314009555c4", "57115ff82419e314009555c5", "57115ff82419e314009555c6", "57115ff82419e314009555c7", "571c8539dd7acb1400e4c0e2", "571c8539dd7acb1400e4c0e3", "571c8539dd7acb1400e4c0e4", "571c8539dd7acb1400e4c0e5", "571c8539dd7acb1400e4c0e6", "571c9348dd7acb1400e4c114", "571c9348dd7acb1400e4c115", "571c9348dd7acb1400e4c116", "571c9348dd7acb1400e4c118", "571cb27fdd7acb1400e4c132", "571cb27fdd7acb1400e4c133", "571cb27fdd7acb1400e4c134", "571cb27fdd7acb1400e4c135", "571cb27fdd7acb1400e4c136", "571cbe35dd7acb1400e4c13c", "571cbe35dd7acb1400e4c13d", "571cbe35dd7acb1400e4c13e", "571cbe35dd7acb1400e4c13f", "571cbe35dd7acb1400e4c140", "571cd3b55efbb31900334e03", "571cd3b55efbb31900334e04", "571cd3b55efbb31900334e05", "571cd3b55efbb31900334e06", "5725bad5271a42140099d0bd", "5725bad5271a42140099d0be", "5725bad5271a42140099d0bf", "5725bad5271a42140099d0c0", "5725bad5271a42140099d0c1", "5726241189a1e219009ac2de", "5726241189a1e219009ac2df", "5726241189a1e219009ac2e0", "5726241189a1e219009ac2e1", "5726241189a1e219009ac2e2", "57264d9edd62a815002e80fe", "57264d9edd62a815002e80ff", "57264d9edd62a815002e8100", "57264d9edd62a815002e8101", "572650325951b619008f6fa9", "572650325951b619008f6faa", "572650325951b619008f6fab", "572659535951b619008f703f", "572659535951b619008f7040", "572659535951b619008f7041", "572659535951b619008f7042", "5725c604271a42140099d185", "5725c604271a42140099d186", "5725c604271a42140099d187", "5725c604271a42140099d188", "5725c604271a42140099d189", "5725d79e89a1e219009abf90", "5725d79e89a1e219009abf91", "5725d79e89a1e219009abf92", "5725d79e89a1e219009abf93", "5725d79e89a1e219009abf94", "5725db4aec44d21400f3d6bd", "5725db4aec44d21400f3d6be", "5725db4aec44d21400f3d6bf", "5725db4aec44d21400f3d6c0", "5725dc1638643c19005ace01", "5725dc1638643c19005ace02", "5725dc1638643c19005ace03", "5725dc1638643c19005ace04", "5725dc1638643c19005ace05", "5725f39638643c19005acef7", "5725f39638643c19005acef8", "5725f39638643c19005acef9", "5725f39638643c19005acefa", "5725f39638643c19005acefb", "5725cc38ec44d21400f3d5bb", "5725cc38ec44d21400f3d5bc", "5725cc38ec44d21400f3d5bd", "5725cc38ec44d21400f3d5be", "5725cc38ec44d21400f3d5bf", "57268e2bf1498d1400e8e3b0", "57268e2bf1498d1400e8e3b1", "57268e2bf1498d1400e8e3b2", "57268e2bf1498d1400e8e3b3", "57268e2bf1498d1400e8e3b4", "572651f9f1498d1400e8dbee", "572651f9f1498d1400e8dbef", "572651f9f1498d1400e8dbf0", "572651f9f1498d1400e8dbf1", "572651f9f1498d1400e8dbf2", "5726938af1498d1400e8e446", "5726938af1498d1400e8e447", "5726938af1498d1400e8e448", "5726938af1498d1400e8e449", "5726938af1498d1400e8e44a", "57265e455951b619008f70bb", "57265e455951b619008f70bc", "57265e455951b619008f70bd", "5726975c708984140094cb1f", "5726975c708984140094cb20", "5726975c708984140094cb21", "5726a299dd62a815002e8b9e", "5726a299dd62a815002e8b9f", "5726a299dd62a815002e8ba0", "5726a299dd62a815002e8ba1", "5726a299dd62a815002e8ba2", "5726a5525951b619008f78dd", "5726a5525951b619008f78de", "5726a5525951b619008f78df", "5726a5525951b619008f78e0", "5726a5525951b619008f78e1", "5725b81b271a42140099d097", "5725b81b271a42140099d098", "5725b81b271a42140099d099", "5725b81b271a42140099d09a", "5725b81b271a42140099d09b", "5728349dff5b5019007d9efe", "5728349dff5b5019007d9eff", "5728349dff5b5019007d9f00", "5728349dff5b5019007d9f01", "5728349dff5b5019007d9f02", "5729e2316aef0514001550c4", "5729e2316aef0514001550c5", "5729e2316aef0514001550c6", "5729e2316aef0514001550c7", "5729e2316aef0514001550c8", "5729feaf6aef051400155188", "5729feaf6aef051400155189", "5729feaf6aef05140015518a", "5729feaf6aef05140015518b", "5729feaf6aef05140015518c", "572a005f1d046914007796b7", "572a005f1d046914007796b8", "572a005f1d046914007796b9", "572a005f1d046914007796bb", "572a005f1d046914007796ba", "572a020f6aef051400155198", "572a020f6aef051400155199", "572a020f6aef05140015519a", "572a020f6aef05140015519b", "572a020f6aef05140015519c", "572a0bebaf94a219006aa76f", "572a0bebaf94a219006aa770", "572a0bebaf94a219006aa771", "572a0bebaf94a219006aa772", "572a0bebaf94a219006aa773", "5725bae289a1e219009abd90", "5725bae289a1e219009abd91", "5725bae289a1e219009abd92", "5725c337271a42140099d163", "5725c337271a42140099d164", "5725c337271a42140099d165", "5725c337271a42140099d166", "5725c337271a42140099d167", "5726400589a1e219009ac5ee", "5726400589a1e219009ac5ef", "5726400589a1e219009ac5f0", "5726400589a1e219009ac5f1", "5726400589a1e219009ac5f2", "5725c91e38643c19005acceb", "5725c91e38643c19005accec", "5725c91e38643c19005acced", "5725c91e38643c19005accee", "5726449f1125e71900ae1928", "5726449f1125e71900ae1929", "5726449f1125e71900ae192a", "5726449f1125e71900ae192b", "572648e8dd62a815002e8076", "572648e8dd62a815002e8077", "572648e8dd62a815002e8078", "572648e8dd62a815002e8079", "57268da7f1498d1400e8e39c", "57268da7f1498d1400e8e39d", "57268da7f1498d1400e8e39e", "57268da7f1498d1400e8e39f", "57268da7f1498d1400e8e3a0", "57269016708984140094ca41", "57269016708984140094ca42", "5725edfe38643c19005ace9f", "5725edfe38643c19005acea0", "5725edfe38643c19005acea1", "5725edfe38643c19005acea2", "5725edfe38643c19005acea3", "5725f00938643c19005aced7", "5725f00938643c19005aced8", "5725f00938643c19005aced9", "5725f00938643c19005aceda", "5725f00938643c19005acedb", "5725f8f5ec44d21400f3d7b1", "5725f8f5ec44d21400f3d7b2", "5725f8f5ec44d21400f3d7b3", "5725f8f5ec44d21400f3d7b4", "5725f8f5ec44d21400f3d7b5", "5725fcbe271a42140099d3ad", "5725fcbe271a42140099d3ae", "5725fcbe271a42140099d3af", "5725fcbe271a42140099d3b0", "5725fe63ec44d21400f3d7dd", "5725fe63ec44d21400f3d7de", "5725fe63ec44d21400f3d7df", "5725fe63ec44d21400f3d7e0", "5726385e271a42140099d797", "5726385e271a42140099d798", "5726385e271a42140099d799", "5726385e271a42140099d79a", "57263c78ec44d21400f3dc7b", "57263c78ec44d21400f3dc7c", "57263c78ec44d21400f3dc7d", "57263c78ec44d21400f3dc7e", "57263ea0271a42140099d7c3", "57263ea0271a42140099d7c4", "57263ea0271a42140099d7c5", "57263ea0271a42140099d7c6", "5726414e271a42140099d7e5", "5726414e271a42140099d7e6", "5726414e271a42140099d7e7", "5726472bdd62a815002e8042", "5726472bdd62a815002e8043", "5726472bdd62a815002e8044", "5726472bdd62a815002e8045", "5726472bdd62a815002e8046", "57264b3edd62a815002e80aa", "57264b3edd62a815002e80ab", "57264b3edd62a815002e80ac", "57264b3edd62a815002e80ad", "57264f18f1498d1400e8dbae", "57264f18f1498d1400e8dbaf", "57264f18f1498d1400e8dbb0", "57264f18f1498d1400e8dbb1", "57264f18f1498d1400e8dbb2", "5726534d708984140094c26d", "5726534d708984140094c26e", "5726534d708984140094c26f", "5726534d708984140094c270", "5726534d708984140094c271", "57265642f1498d1400e8dc68", "57265642f1498d1400e8dc69", "57265642f1498d1400e8dc6b", "57265642f1498d1400e8dc6c", "57265642f1498d1400e8dc6a", "5726577f708984140094c301", "5726577f708984140094c302", "5726577f708984140094c303", "5726577f708984140094c304", "572663a9f1498d1400e8ddf2", "572663a9f1498d1400e8ddf4", "572663a9f1498d1400e8ddf5", "572663a9f1498d1400e8ddf6", "572669a9dd62a815002e8416", "572669a9dd62a815002e8417", "572669a9dd62a815002e8418", "572669a9dd62a815002e8419", "572669a9dd62a815002e841a", "57268066708984140094c821", "57268066708984140094c822", "57268066708984140094c823", "57268066708984140094c824", "57268066708984140094c825", "572683f95951b619008f7525", "572683f95951b619008f7526", "572683f95951b619008f7527", "572683f95951b619008f7528", "572683f95951b619008f7529", "57268527708984140094c8bf", "57268527708984140094c8c0", "57268527708984140094c8c1", "57268527708984140094c8c2", "5726660d5951b619008f71b1", "5726660d5951b619008f71b2", "5726660d5951b619008f71b3", "5726660d5951b619008f71b4", "5726660d5951b619008f71b5", "572671e55951b619008f72d7", "572671e55951b619008f72d8", "572671e55951b619008f72d9", "572671e55951b619008f72da", "572671e55951b619008f72db", "5726847f708984140094c8ab", "5726847f708984140094c8ac", "5726847f708984140094c8ad", "5726847f708984140094c8ae", "5726847f708984140094c8af", "57268a8fdd62a815002e88ce", "57268a8fdd62a815002e88cf", "57268a8fdd62a815002e88d0", "57268a8fdd62a815002e88d1", "57268a8fdd62a815002e88d2", "57269698dd62a815002e8a6c", "57269698dd62a815002e8a6d", "57269698dd62a815002e8a6e", "57269698dd62a815002e8a6f", "57269698dd62a815002e8a70", "572685cd5951b619008f7573", "572685cd5951b619008f7574", "572685cd5951b619008f7575", "572685cd5951b619008f7576", "572685cd5951b619008f7577", "5726bf325951b619008f7cfd", "5726bf325951b619008f7cfe", "5726bf325951b619008f7cff", "5726bf325951b619008f7d00", "5726bf325951b619008f7d01", "5726e37ef1498d1400e8eed8", "5726e37ef1498d1400e8eed9", "5726e37ef1498d1400e8eeda", "5726e37ef1498d1400e8eedb", "5726e37ef1498d1400e8eedc", "5726f1ec708984140094d6aa", "5726f1ec708984140094d6ad", "5726f1ec708984140094d6ab", "5726f1ec708984140094d6ac", "5726f1ec708984140094d6a9", "5726f4a0708984140094d6e9", "5726f4a0708984140094d6ea", "5726f4a0708984140094d6eb", "5726f4a0708984140094d6ec", "5726f4a0708984140094d6ed", "572734af708984140094dae3", "572734af708984140094dae4", "572734af708984140094dae5", "572734af708984140094dae6", "572735a15951b619008f86bf", "572735a15951b619008f86c0", "572735a15951b619008f86c1", "572735a15951b619008f86c2", "57273f9d708984140094db51", "57273f9d708984140094db52", "57273f9d708984140094db53", "57273f9d708984140094db54", "572743fb708984140094db93", "572743fb708984140094db94", "572743fb708984140094db95", "57276166dd62a815002e9bd8", "57276166dd62a815002e9bd9", "57276166dd62a815002e9bda", "57276166dd62a815002e9bdb", "5726a8d4dd62a815002e8c34", "5726a8d4dd62a815002e8c35", "5726a8d4dd62a815002e8c36", "5726a8d4dd62a815002e8c37", "5726a8d4dd62a815002e8c38", "5726acc1f1498d1400e8e6ca", "5726acc1f1498d1400e8e6cb", "5726acc1f1498d1400e8e6cc", "5726acc1f1498d1400e8e6cd", "5726acc1f1498d1400e8e6ce", "57273455f1498d1400e8f48c", "57273455f1498d1400e8f48d", "57273455f1498d1400e8f48e", "57273455f1498d1400e8f48f", "57273455f1498d1400e8f490", "57273dccdd62a815002e99fa", "57273dccdd62a815002e99fb", "57273dccdd62a815002e99fc", "57273dccdd62a815002e99fd", "572754cd5951b619008f8863", "572754cd5951b619008f8864", "572754cd5951b619008f8865", "572754cd5951b619008f8866", "572754cd5951b619008f8867", "5726da89dd62a815002e92b2", "5726da89dd62a815002e92b3", "5726da89dd62a815002e92b4", "5726da89dd62a815002e92b5", "5726da89dd62a815002e92b6", "5726ddf6f1498d1400e8ee04", "5726ddf6f1498d1400e8ee05", "5726ddf6f1498d1400e8ee06", "5726ddf6f1498d1400e8ee08", "5726ddf6f1498d1400e8ee07", "5726e08e5951b619008f810f", "5726e08e5951b619008f8110", "5726e08e5951b619008f8111", "5726e08e5951b619008f8112", "5726e08e5951b619008f8113", "5726ea985951b619008f8261", "5726ea985951b619008f8262", "5726ea985951b619008f8263", "5726ea985951b619008f8264", "5726ea985951b619008f8265", "5726f48df1498d1400e8f0da", "5726f48df1498d1400e8f0db", "5726f48df1498d1400e8f0dc", "5726f48df1498d1400e8f0dd", "5726f48df1498d1400e8f0de", "5726eb8bf1498d1400e8efe2", "5726eb8bf1498d1400e8efe3", "5726eb8bf1498d1400e8efe4", "5726eb8bf1498d1400e8efe5", "5726eb8bf1498d1400e8efe6", "5728f2e26aef051400154896", "5728f2e26aef051400154897", "5728f2e26aef051400154898", "5728f2e26aef051400154899", "572a03086aef0514001551a2", "572a03086aef0514001551a3", "572a03086aef0514001551a4", "572a03086aef0514001551a5", "572a04d51d046914007796cd", "572a04d51d046914007796ce", "572a04d51d046914007796cf", "572a04d51d046914007796d0", "572a07fc6aef0514001551dc", "572a07fc6aef0514001551dd", "572a07fc6aef0514001551de", "572a07fc6aef0514001551df", "572a0e4b6aef051400155214", "572a0e4b6aef051400155215", "572a0e4b6aef051400155216", "572a0e4b6aef051400155217"], "shots": [["पैंथर्स डिफ़ेंस ने कितने अंक दिए?", "308"], ["जेरेड एलन के पास कितने करियर सैक थे?", "136"], ["ल्यूक कुएक्ली ने कितने टैकल रजिस्टर किए?", "118"]], "retrieved": {"5727213c708984140094da35": [140, 804, 173, 427, 396], "5727213c708984140094da36": [140, 776, 19, 33, 882], "5727213c708984140094da38": [140, 750, 141, 880, 60], "5727213c708984140094da39": [140, 141, 750, 804, 694], "5727213c708984140094da37": [140, 108, 427, 717, 15], "5728d63c4b864d1900164f18": [140, 717, 457, 776, 396], "5728d63c4b864d1900164f19": [140, 318, 19, 227, 457], "5728d63c4b864d1900164f1a": [140, 236, 109, 223, 100], "5728d63c4b864d1900164f1c": [141, 140, 750, 440, 144], "5728d63c4b864d1900164f1b": [141, 140, 142, 440, 580], "5728202c4b864d19001644ec": [142, 141, 804, 596, 874], "5728202c4b864d19001644ed": [141, 142, 440, 143, 140], "5728202c4b864d19001644ee": [141, 142, 140, 840, 440], "5728202c4b864d19001644ef": [141, 142, 140, 143, 477], "5728202c4b864d19001644f0": [141, 142, 143, 776, 477], "5728e07e3acd2414000e00e9": [141, 142, 143, 776, 140], "5728e07e3acd2414000e00ea": [141, 143, 142, 144, 767], "5728e07e3acd2414000e00eb": [141, 31, 142, 767, 143], "5728e07e3acd2414000e00ec": [141, 142, 143, 202, 767], "5728e07e3acd2414000e00ed": [141, 142, 143, 140, 144], "57282dfb4b864d1900164668": [142, 388, 439, 149, 216], "57282dfb4b864d1900164669": [142, 348, 667, 390, 587], "57282dfb4b864d190016466a": [142, 348, 587, 667, 463], "57282dfb4b864d190016466b": [443, 699, 628, 882, 434], "57282dfb4b864d190016466c": [699, 617, 294, 603, 624], "5728e715ff5b5019007da914": [142, 141, 143, 216, 800], "5728e715ff5b5019007da916": [443, 782, 800, 136, 434], "5728e715ff5b5019007da917": [634, 318, 346, 445, 591], "5728e715ff5b5019007da918": [430, 434, 722, 318, 252], "5728eb1a3acd2414000e01c5": [143, 141, 434, 142, 321], "5728eb1a3acd2414000e01c6": [143, 144, 528, 434, 326], "5728eb1a3acd2414000e01c7": [840, 882, 228, 443, 318], "5728eb1a3acd2414000e01c8": [219, 321, 248, 143, 591], "5728eb1a3acd2414000e01c9": [143, 144, 141, 304, 593], "5728eef92ca10214002daab0": [143, 141, 142, 144, 25], "5728eef92ca10214002daab1": [144, 149, 44, 122, 767], "5728eef92ca10214002daab2": [144, 596, 445, 346, 326], "5728eef92ca10214002daab3": [144, 141, 142, 842, 767], "5728eef92ca10214002daab4": [844, 144, 788, 286, 270], "57273a465951b619008f86ff": [145, 148, 149, 147, 175], "57273a465951b619008f8700": [145, 296, 69, 820, 147], "57273a465951b619008f8701": [145, 820, 870, 627, 872], "57273a465951b619008f8702": [145, 149, 767, 147, 820], "57273a465951b619008f8703": [145, 147, 148, 149, 146], "57273e50dd62a815002e9a02": [146, 684, 310, 501, 509], "57273e50dd62a815002e9a03": [146, 145, 820, 148, 147], "57273e50dd62a815002e9a04": [146, 9, 193, 512, 96], "57273e50dd62a815002e9a05": [146, 820, 484, 58, 258], "57273f27dd62a815002e9a0a": [147, 202, 820, 767, 145], "57273f27dd62a815002e9a0b": [147, 603, 239, 202, 153], "57273f27dd62a815002e9a0c": [147, 148, 145, 820, 146], "57273f27dd62a815002e9a0d": [145, 148, 147, 149, 146], "57274e0d708984140094dbe5": [148, 236, 820, 176, 367], "57274e0d708984140094dbe6": [148, 820, 147, 146, 475], "57274e0d708984140094dbe7": [148, 145, 820, 147, 490], "57274e0d708984140094dbe8": [148, 145, 147, 149, 279], "572750e8dd62a815002e9af0": [149, 145, 296, 490, 435], "572750e8dd62a815002e9af1": [149, 145, 194, 294, 771], "572750e8dd62a815002e9af2": [149, 215, 542, 717, 236], "572750e8dd62a815002e9af3": [149, 542, 236, 321, 477], "572750e8dd62a815002e9af4": [149, 145, 148, 386, 212], "57274b35f1498d1400e8f5d4": [150, 151, 152, 153, 321], "57274b35f1498d1400e8f5d5": [150, 152, 25, 153, 177], "57274b35f1498d1400e8f5d6": [150, 165, 466, 223, 423], "57274b35f1498d1400e8f5d7": [150, 225, 227, 223, 220], "57274beff1498d1400e8f5e4": [151, 150, 152, 153, 501], "57274beff1498d1400e8f5e5": [151, 484, 275, 501, 453], "57274beff1498d1400e8f5e6": [151, 501, 855, 453, 774], "57274beff1498d1400e8f5e7": [151, 321, 826, 150, 25], "5727515f708984140094dc11": [152, 153, 25, 154, 151], "5727515f708984140094dc12": [152, 153, 157, 25, 154], "5727515f708984140094dc13": [152, 153, 154, 25, 151], "5727515f708984140094dc14": [152, 153, 151, 150, 154], "5727515f708984140094dc15": [152, 225, 157, 176, 153], "57275409708984140094dc35": [153, 154, 152, 601, 321], "57275409708984140094dc36": [153, 152, 321, 695, 150], "57275409708984140094dc37": [153, 152, 259, 826, 321], "572757bef1498d1400e8f690": [154, 153, 25, 152, 151], "572757bef1498d1400e8f691": [154, 153, 804, 321, 152], "572757bef1498d1400e8f692": [154, 225, 52, 53, 161], "572757bef1498d1400e8f693": [154, 226, 157, 225, 52], "572757bef1498d1400e8f694": [154, 153, 152, 27, 321], "5727aec03acd2414000de991": [155, 156, 157, 158, 161], "5727aec03acd2414000de992": [151, 826, 150, 153, 152], "5727aec03acd2414000de993": [155, 156, 51, 206, 205], "5727aec03acd2414000de994": [151, 177, 826, 695, 321], "5727aec03acd2414000de995": [155, 156, 176, 175, 113], "5727cc15ff5b5019007d9576": [156, 155, 157, 331, 445], "5727cc15ff5b5019007d9577": [156, 161, 110, 367, 51], "5727cc15ff5b5019007d9578": [156, 155, 157, 176, 114], "5727d0f73acd2414000ded13": [157, 154, 152, 153, 17], "5727d0f73acd2414000ded14": [157, 154, 158, 738, 321], "5727d0f73acd2414000ded15": [157, 231, 81, 627, 158], "5727d1c93acd2414000ded3f": [158, 259, 604, 686, 826], "5727d1c93acd2414000ded40": [158, 157, 118, 171, 70], "5727d1c93acd2414000ded41": [158, 157, 118, 171, 4], "5727d1c93acd2414000ded42": [158, 98, 176, 153, 383], "5727de862ca10214002d9860": [157, 159, 416, 179, 571], "5727de862ca10214002d9861": [571, 159, 157, 187, 158], "5727de862ca10214002d9862": [157, 179, 159, 187, 367], "5727de862ca10214002d9863": [157, 159, 158, 52, 179], "5727c94bff5b5019007d954a": [160, 161, 90, 94, 163], "5727c94bff5b5019007d954b": [160, 163, 161, 164, 51], "5727c94bff5b5019007d954c": [160, 161, 163, 164, 51], "5727c94bff5b5019007d954d": [160, 161, 163, 164, 51], "5727c94bff5b5019007d954e": [160, 161, 163, 164, 110], "5727cb4b2ca10214002d9676": [161, 164, 160, 163, 51], "5727cb4b2ca10214002d9677": [161, 164, 160, 163, 349], "5727cb4b2ca10214002d9678": [231, 162, 52, 161, 49], "5727cb4b2ca10214002d9679": [161, 160, 162, 163, 10], "5727cb4b2ca10214002d967a": [161, 160, 163, 164, 110], "57280fd3ff5b5019007d9c26": [162, 161, 51, 110, 119], "57280fd3ff5b5019007d9c27": [161, 163, 164, 160, 162], "57280fd3ff5b5019007d9c28": [162, 234, 233, 52, 14], "57280fd3ff5b5019007d9c29": [162, 161, 233, 14, 234], "57280fd3ff5b5019007d9c2a": [162, 127, 232, 176, 449], "57281ab63acd2414000df493": [163, 367, 35, 294, 820], "57281ab63acd2414000df494": [163, 160, 161, 164, 51], "57281ab63acd2414000df495": [811, 35, 281, 321, 312], "57281ab63acd2414000df496": [163, 164, 161, 160, 820], "57281ab63acd2414000df497": [163, 164, 161, 160, 126], "572824f13acd2414000df58f": [164, 163, 160, 161, 207], "572824f13acd2414000df590": [164, 207, 73, 162, 208], "572824f13acd2414000df591": [164, 163, 161, 160, 599], "572824f13acd2414000df592": [164, 63, 62, 370, 57], "572824f13acd2414000df593": [164, 163, 161, 160, 236], "572a1046af94a219006aa78d": [165, 168, 167, 166, 223], "572a1046af94a219006aa78e": [167, 165, 168, 166, 169], "572a1046af94a219006aa78f": [165, 179, 1, 199, 86], "572a1046af94a219006aa790": [168, 167, 165, 166, 169], "572a1046af94a219006aa791": [165, 168, 169, 166, 167], "572a11663f37b31900478693": [166, 167, 168, 165, 667], "572a11663f37b31900478694": [166, 165, 169, 168, 627], "572a11663f37b31900478695": [166, 165, 52, 179, 98], "572a11663f37b31900478696": [166, 167, 168, 165, 169], "572a11663f37b31900478697": [166, 294, 216, 184, 767], "572a13841d0469140077973b": [167, 168, 166, 165, 565], "572a13841d0469140077973c": [167, 168, 169, 166, 165], "572a13841d0469140077973d": [167, 22, 236, 194, 20], "572a13841d0469140077973e": [167, 226, 173, 102, 194], "572a13841d0469140077973f": [167, 166, 165, 168, 163], "572a18a4af94a219006aa7df": [168, 167, 165, 166, 169], "572a18a4af94a219006aa7e0": [168, 165, 223, 167, 627], "572a18a4af94a219006aa7e1": [168, 321, 146, 185, 169], "572a18a4af94a219006aa7e2": [169, 168, 66, 490, 165], "572a18a4af94a219006aa7e3": [168, 169, 165, 166, 167], "572a1a5c6aef051400155284": [169, 294, 165, 767, 216], "572a1a5c6aef051400155285": [169, 165, 168, 166, 767], "572a1a5c6aef051400155286": [169, 166, 168, 767, 165], "572a1a5c6aef051400155287": [159, 440, 70, 600, 218], "572a1a5c6aef051400155288": [169, 168, 167, 165, 294], "5727f3193acd2414000df0a5": [170, 174, 172, 171, 339], "5727f3193acd2414000df0a6": [170, 174, 171, 172, 173], "5727f3193acd2414000df0a7": [64, 170, 424, 342, 339], "5727f3193acd2414000df0a8": [170, 174, 172, 744, 85], "5727f3193acd2414000df0a9": [170, 174, 183, 753, 171], "57280f0d3acd2414000df35b": [174, 172, 171, 173, 312], "57280f0d3acd2414000df35c": [174, 171, 183, 172, 170], "57280f0d3acd2414000df35d": [174, 171, 173, 183, 100], "57280f0d3acd2414000df35e": [171, 170, 235, 100, 477], "57280f0d3acd2414000df35f": [174, 171, 489, 520, 817], "572811434b864d190016438c": [172, 174, 170, 553, 171], "572811434b864d190016438d": [172, 171, 170, 174, 131], "572811434b864d190016438e": [172, 170, 171, 131, 51], "572811434b864d190016438f": [172, 780, 454, 508, 677], "572811434b864d1900164390": [172, 122, 174, 444, 878], "57281a952ca10214002d9dea": [174, 173, 172, 171, 312], "57281a952ca10214002d9deb": [173, 171, 174, 172, 70], "57281a952ca10214002d9dec": [173, 174, 171, 223, 15], "57281a952ca10214002d9ded": [173, 174, 171, 172, 170], "57281a952ca10214002d9dee": [173, 174, 172, 171, 878], "572828383acd2414000df5c3": [174, 172, 173, 171, 170], "572828383acd2414000df5c4": [174, 173, 171, 124, 172], "572828383acd2414000df5c5": [174, 171, 183, 172, 173], "572828383acd2414000df5c6": [174, 124, 173, 5, 123], "572828383acd2414000df5c7": [174, 172, 171, 170, 677], "57284b904b864d19001648e2": [175, 114, 176, 367, 855], "57284b904b864d19001648e3": [175, 59, 102, 0, 674], "57284b904b864d19001648e4": [175, 70, 176, 182, 236], "57284b904b864d19001648e5": [175, 367, 60, 164, 110], "57284b904b864d19001648e6": [175, 367, 176, 163, 115], "57284d484b864d1900164900": [176, 367, 163, 5, 115], "57284d484b864d1900164901": [176, 175, 186, 70, 98], "57284d484b864d1900164902": [176, 175, 115, 367, 5], "57284d484b864d1900164903": [176, 175, 5, 159, 51], "57284d484b864d1900164904": [176, 115, 367, 113, 5], "572855973acd2414000df925": [177, 501, 151, 150, 453], "572855973acd2414000df926": [177, 855, 98, 176, 175], "572855973acd2414000df927": [177, 27, 155, 855, 156], "572855973acd2414000df928": [259, 321, 855, 151, 177], "572855973acd2414000df929": [177, 156, 155, 51, 564], "57286951ff5b5019007da20e": [178, 175, 179, 98, 177], "57286951ff5b5019007da20f": [178, 16, 179, 223, 190], "57286951ff5b5019007da210": [178, 179, 466, 159, 845], "57286951ff5b5019007da211": [178, 179, 177, 175, 176], "57286951ff5b5019007da212": [178, 16, 69, 115, 122], "572870b2ff5b5019007da222": [179, 275, 223, 178, 114], "572870b2ff5b5019007da223": [179, 159, 229, 225, 700], "572870b2ff5b5019007da224": [179, 223, 178, 165, 159], "572870b2ff5b5019007da225": [179, 52, 157, 178, 171], "57286dfa2ca10214002da332": [180, 182, 360, 612, 627], "57286dfa2ca10214002da333": [180, 234, 186, 126, 639], "57286dfa2ca10214002da334": [180, 181, 182, 129, 233], "57286dfa2ca10214002da335": [180, 128, 129, 126, 233], "57286dfa2ca10214002da336": [180, 128, 129, 126, 618], "57286fa83acd2414000df9e5": [181, 125, 560, 108, 717], "57286fa83acd2414000df9e6": [181, 128, 52, 180, 162], "57286fa83acd2414000df9e7": [181, 182, 52, 885, 128], "57286fa83acd2414000df9e8": [674, 180, 882, 181, 416], "57286fa83acd2414000df9e9": [181, 128, 126, 54, 233], "57287d4a2ca10214002da3e4": [182, 180, 217, 183, 181], "57287d4a2ca10214002da3e5": [182, 180, 116, 181, 183], "57287d4a2ca10214002da3e6": [180, 182, 183, 159, 190], "57287d4a2ca10214002da3e7": [182, 180, 186, 8, 17], "57287d4a2ca10214002da3e8": [182, 180, 186, 360, 159], "572881d34b864d1900164a5a": [183, 880, 53, 776, 50], "572881d34b864d1900164a5b": [183, 49, 176, 115, 114], "572881d34b864d1900164a5c": [183, 49, 176, 114, 115], "572881d34b864d1900164a5d": [183, 180, 171, 182, 116], "572881d34b864d1900164a5e": [183, 131, 182, 795, 136], "5728848cff5b5019007da298": [184, 182, 419, 222, 223], "5728848cff5b5019007da299": [184, 182, 419, 70, 225], "5728848cff5b5019007da29a": [795, 180, 182, 116, 184], "5728848cff5b5019007da29b": [128, 129, 180, 126, 441], "57290b21af94a219006a9fcf": [185, 188, 189, 187, 182], "57290b21af94a219006a9fd0": [185, 22, 722, 627, 202], "57290b21af94a219006a9fd1": [185, 187, 188, 189, 440], "57290b21af94a219006a9fd2": [185, 440, 187, 188, 182], "57290ee2af94a219006a9fff": [186, 75, 180, 182, 125], "57290ee2af94a219006aa000": [186, 671, 321, 281, 612], "57290ee2af94a219006aa001": [187, 186, 685, 622, 159], "57290ee2af94a219006aa002": [0, 774, 762, 186, 18], "57290ee2af94a219006aa003": [336, 701, 273, 699, 242], "572914f46aef051400154a46": [618, 315, 795, 632, 159], "572914f46aef051400154a47": [187, 361, 159, 325, 303], "572914f46aef051400154a48": [187, 159, 186, 189, 233], "572914f46aef051400154a49": [187, 159, 189, 188, 127], "572920d73f37b31900478055": [188, 249, 321, 45, 26], "572920d73f37b31900478056": [188, 185, 189, 774, 36], "572920d73f37b31900478057": [188, 189, 795, 560, 185], "572920d73f37b31900478058": [188, 542, 603, 294, 475], "5729281baf94a219006aa11f": [189, 188, 38, 566, 602], "5729281baf94a219006aa120": [189, 237, 566, 4, 0], "5729281baf94a219006aa121": [189, 566, 312, 359, 578], "5729281baf94a219006aa122": [189, 566, 187, 185, 293], "57293bc91d0469140077919b": [190, 194, 192, 159, 709], "57293bc91d0469140077919c": [190, 51, 54, 178, 312], "57293bc91d0469140077919d": [190, 194, 192, 600, 709], "57293bc91d0469140077919e": [190, 194, 192, 98, 70], "57293bc91d0469140077919f": [190, 469, 194, 192, 167], "57293e221d046914007791d5": [191, 321, 68, 617, 587], "57293e221d046914007791d6": [191, 64, 376, 356, 68], "57293e221d046914007791d7": [191, 83, 84, 62, 63], "57293e221d046914007791d8": [191, 194, 356, 237, 235], "57293e221d046914007791d9": [191, 193, 208, 207, 62], "572940973f37b319004781a5": [192, 194, 190, 431, 528], "572940973f37b319004781a6": [190, 192, 194, 528, 270], "572940973f37b319004781a7": [443, 192, 575, 100, 427], "572940973f37b319004781a8": [192, 593, 193, 492, 168], "57294209af94a219006aa201": [193, 192, 208, 101, 191], "57294209af94a219006aa202": [193, 789, 176, 140, 223], "57294209af94a219006aa203": [193, 127, 205, 121, 176], "57294209af94a219006aa204": [193, 176, 279, 98, 220], "57294e6b1d04691400779275": [194, 236, 162, 673, 14], "57294e6b1d04691400779276": [194, 70, 77, 96, 463], "57294e6b1d04691400779277": [194, 190, 185, 192, 219], "572957ad1d046914007792d9": [195, 138, 199, 196, 88], "572957ad1d046914007792da": [195, 86, 88, 197, 138], "572957ad1d046914007792db": [197, 196, 199, 198, 195], "572957ad1d046914007792dc": [199, 195, 427, 229, 98], "572957ad1d046914007792dd": [195, 477, 197, 675, 102], "5729582b1d046914007792e3": [196, 197, 235, 100, 102], "5729582b1d046914007792e4": [196, 197, 198, 199, 195], "5729582b1d046914007792e5": [196, 717, 676, 699, 542], "5729582b1d046914007792e6": [196, 197, 198, 782, 195], "5729582b1d046914007792e7": [196, 51, 108, 80, 53], "572968cf1d046914007793cb": [197, 196, 198, 199, 195], "572968cf1d046914007793cc": [197, 196, 198, 199, 195], "572968cf1d046914007793cd": [197, 173, 207, 60, 101], "572968cf1d046914007793ce": [197, 100, 173, 103, 102], "572968cf1d046914007793cf": [197, 100, 173, 102, 195], "57296b151d046914007793f1": [198, 197, 196, 195, 199], "57296b151d046914007793f2": [198, 89, 88, 138, 464], "57296b151d046914007793f3": [198, 197, 196, 195, 199], "57296b151d046914007793f4": [198, 197, 196, 195, 199], "57296d1b1d0469140077940d": [199, 198, 196, 197, 195], "57296d1b1d0469140077940e": [199, 198, 195, 396, 197], "57296d1b1d0469140077940f": [199, 195, 196, 197, 198], "57296d571d04691400779413": [200, 204, 202, 201, 203], "57296d571d04691400779414": [200, 202, 201, 203, 204], "57296d571d04691400779415": [200, 203, 202, 201, 204], "57296d571d04691400779416": [200, 202, 201, 204, 203], "57296d571d04691400779417": [203, 200, 204, 767, 195], "57296f293f37b319004783a3": [200, 201, 203, 204, 202], "57296f293f37b319004783a4": [201, 202, 200, 650, 102], "57296f293f37b319004783a5": [201, 650, 102, 464, 64], "57296f293f37b319004783a6": [201, 650, 102, 64, 202], "57296f293f37b319004783a7": [201, 200, 202, 470, 203], "572991943f37b319004784a1": [202, 200, 203, 204, 201], "572991943f37b319004784a2": [202, 200, 203, 201, 204], "572991943f37b319004784a3": [202, 200, 201, 204, 203], "572991943f37b319004784a4": [200, 203, 202, 204, 201], "572991943f37b319004784a5": [202, 200, 203, 201, 204], "57299a6f6aef051400155016": [203, 200, 201, 204, 202], "57299a6f6aef051400155017": [203, 204, 200, 202, 722], "57299a6f6aef051400155019": [203, 95, 21, 121, 861], "57299a6f6aef05140015501a": [203, 200, 201, 204, 202], "57299a6f6aef051400155018": [203, 200, 201, 202, 204], "5729a26d6aef05140015505a": [204, 200, 22, 201, 202], "5729a26d6aef05140015505b": [204, 237, 235, 825, 356], "5729a26d6aef05140015505c": [204, 203, 202, 236, 820], "5729a26d6aef05140015505d": [200, 203, 202, 204, 201], "572f6a0ba23a5019007fc5eb": [205, 549, 21, 441, 662], "572f6a0ba23a5019007fc5ec": [205, 159, 441, 102, 335], "572f6a0ba23a5019007fc5ed": [205, 112, 237, 59, 306], "572f6a0ba23a5019007fc5ee": [205, 206, 51, 111, 5], "572ff12e04bcaa1900d76eff": [205, 549, 21, 441, 662], "572ff12e04bcaa1900d76f00": [205, 206, 207, 208, 549], "572ff12e04bcaa1900d76f02": [205, 112, 206, 179, 587], "572ff12e04bcaa1900d76f03": [205, 206, 75, 51, 587], "572ff673b2c2fd1400568669": [206, 491, 607, 101, 80], "572ff673b2c2fd140056866a": [206, 208, 101, 110, 30], "572ff673b2c2fd140056866b": [206, 491, 395, 54, 787], "572ffd75b2c2fd14005686e5": [207, 208, 192, 193, 164], "572ffd75b2c2fd14005686e6": [207, 192, 266, 164, 109], "572ffd75b2c2fd14005686e7": [207, 208, 205, 57, 206], "572ffd75b2c2fd14005686e8": [207, 164, 208, 62, 193], "572ffd75b2c2fd14005686e9": [207, 208, 41, 164, 335], "572fffb404bcaa1900d76fef": [207, 208, 193, 75, 61], "572fffb404bcaa1900d76ff0": [208, 356, 207, 84, 164], "572fffb404bcaa1900d76ff1": [207, 208, 101, 75, 164], "572fffb404bcaa1900d76ff2": [208, 61, 115, 83, 73], "572fffb404bcaa1900d76ff3": [207, 208, 103, 101, 75], "57300a9a04bcaa1900d77063": [209, 75, 9, 112, 51], "57300a9a04bcaa1900d77064": [209, 610, 220, 75, 293], "57300a9a04bcaa1900d77065": [209, 214, 795, 158, 252], "57300a9a04bcaa1900d77066": [209, 233, 75, 220, 79], "57300a9a04bcaa1900d77067": [209, 234, 127, 13, 233], "572fadcbb2c2fd1400568329": [210, 214, 49, 75, 79], "572fadcbb2c2fd140056832a": [210, 214, 211, 212, 76], "572fadcbb2c2fd140056832c": [210, 214, 213, 212, 211], "572fadcbb2c2fd140056832b": [210, 214, 212, 213, 176], "572fadcbb2c2fd140056832d": [210, 214, 231, 213, 618], "572fc6f204bcaa1900d76cf5": [211, 213, 212, 210, 214], "572fc6f204bcaa1900d76cf6": [211, 212, 213, 214, 77], "572fc6f204bcaa1900d76cf7": [294, 434, 519, 490, 211], "572fc6f204bcaa1900d76cf8": [211, 70, 212, 434, 420], "572fc6f204bcaa1900d76cf9": [211, 212, 213, 214, 76], "572fc8a904bcaa1900d76d1f": [212, 393, 43, 151, 98], "572fc8a904bcaa1900d76d20": [212, 603, 774, 151, 149], "572fc8a904bcaa1900d76d21": [149, 844, 490, 872, 693], "572fc8a904bcaa1900d76d22": [212, 77, 99, 98, 70], "572fcc43b2c2fd140056847d": [213, 211, 212, 214, 210], "572fcc43b2c2fd140056847e": [213, 388, 321, 670, 624], "572fcc43b2c2fd140056847f": [213, 132, 795, 774, 591], "572fcc43b2c2fd1400568480": [213, 95, 132, 553, 235], "572fdb17b2c2fd140056851d": [214, 98, 229, 820, 43], "572fdb17b2c2fd140056851e": [214, 51, 79, 46, 77], "572fdb17b2c2fd140056851f": [214, 312, 348, 484, 229], "572fdb17b2c2fd1400568520": [214, 226, 555, 142, 600], "572fdb17b2c2fd1400568521": [214, 284, 142, 596, 774], "572ff932a23a5019007fcbd5": [215, 216, 219, 217, 819], "572ff932a23a5019007fcbd6": [215, 216, 219, 452, 419], "572ff932a23a5019007fcbd7": [215, 219, 819, 19, 235], "572ff932a23a5019007fcbd8": [215, 216, 219, 819, 767], "572ff932a23a5019007fcbd9": [215, 216, 819, 452, 245], "572ffe6fb2c2fd14005686ef": [216, 215, 446, 819, 245], "572ffe6fb2c2fd14005686f2": [216, 215, 219, 819, 811], "572ffe6fb2c2fd14005686f0": [216, 596, 150, 681, 321], "572ffe6fb2c2fd14005686f3": [216, 143, 141, 419, 321], "572ffe6fb2c2fd14005686f1": [216, 306, 294, 432, 767], "572ffee1947a6a140053cf14": [217, 32, 30, 225, 220], "572ffee1947a6a140053cf15": [217, 225, 190, 220, 226], "572ffee1947a6a140053cf17": [217, 681, 126, 863, 441], "572ffee1947a6a140053cf16": [217, 772, 427, 301, 863], "572ffee1947a6a140053cf18": [217, 427, 32, 30, 173], "57302700a23a5019007fce89": [218, 49, 181, 182, 341], "57302700a23a5019007fce8a": [218, 219, 159, 341, 215], "57302700a23a5019007fce8b": [218, 13, 14, 54, 70], "57302700a23a5019007fce8c": [218, 165, 179, 167, 363], "57302700a23a5019007fce8d": [218, 579, 538, 175, 427], "57302bd0b2c2fd14005689db": [219, 215, 216, 218, 452], "57302bd0b2c2fd14005689dc": [219, 215, 218, 159, 452], "57302bd0b2c2fd14005689de": [219, 215, 218, 446, 104], "57302bd0b2c2fd14005689df": [219, 215, 216, 104, 166], "57302bd0b2c2fd14005689dd": [219, 215, 218, 556, 168], "573088da069b53140083216b": [220, 54, 179, 231, 301], "573088da069b53140083216c": [112, 220, 161, 301, 178], "573088da069b53140083216d": [220, 179, 108, 223, 236], "573088da069b53140083216e": [220, 223, 419, 222, 224], "573088da069b53140083216f": [220, 48, 13, 41, 53], "57308f6b8ab72b1400f9c580": [221, 223, 109, 106, 208], "57308f6b8ab72b1400f9c581": [221, 108, 789, 776, 882], "57308f6b8ab72b1400f9c582": [221, 789, 223, 224, 109], "57308f6b8ab72b1400f9c583": [221, 224, 223, 49, 222], "57308f6b8ab72b1400f9c584": [221, 223, 167, 224, 882], "573093598ab72b1400f9c5ae": [222, 223, 220, 419, 221], "573093598ab72b1400f9c5af": [222, 223, 419, 221, 220], "573093598ab72b1400f9c5b0": [222, 12, 171, 163, 223], "573093598ab72b1400f9c5b1": [222, 223, 220, 182, 184], "573093598ab72b1400f9c5b2": [222, 182, 701, 419, 220], "57309ef18ab72b1400f9c600": [223, 220, 222, 165, 221], "57309ef18ab72b1400f9c602": [223, 222, 220, 182, 167], "57309ef18ab72b1400f9c601": [223, 173, 11, 233, 115], "57309ef18ab72b1400f9c603": [223, 220, 222, 221, 167], "57309ef18ab72b1400f9c604": [223, 220, 221, 419, 222], "5730a4d02461fd1900a9cf29": [224, 162, 14, 231, 219], "5730a4d02461fd1900a9cf2a": [224, 234, 162, 52, 223], "5730a4d02461fd1900a9cf2d": [224, 50, 52, 220, 231], "5730a4d02461fd1900a9cf2b": [224, 52, 162, 220, 234], "5730a4d02461fd1900a9cf2c": [224, 52, 231, 220, 234], "57309921396df919000961f6": [225, 227, 229, 226, 228], "57309921396df919000961f7": [225, 229, 227, 226, 228], "57309921396df919000961f8": [225, 229, 226, 227, 228], "5730b2312461fd1900a9cfad": [226, 85, 89, 87, 88], "5730b2312461fd1900a9cfae": [226, 217, 98, 227, 79], "5730b2312461fd1900a9cfaf": [226, 159, 685, 885, 173], "5730b2ac2461fd1900a9cfb3": [225, 228, 227, 226, 229], "5730b2ac2461fd1900a9cfb6": [227, 225, 226, 229, 228], "5730b2ac2461fd1900a9cfb4": [227, 463, 33, 102, 579], "5730b2ac2461fd1900a9cfb5": [227, 229, 225, 228, 226], "5730b9852461fd1900a9cff9": [228, 318, 227, 477, 226], "5730b9852461fd1900a9cffa": [228, 882, 284, 585, 227], "5730b9852461fd1900a9cffb": [228, 227, 226, 216, 31], "5730d26fb54a4f140068cc94": [229, 225, 227, 131, 100], "5730d26fb54a4f140068cc95": [229, 133, 130, 28, 579], "5730d26fb54a4f140068cc96": [229, 226, 830, 281, 372], "5730d26fb54a4f140068cc97": [229, 97, 95, 765, 736], "5733d4c8d058e614000b6353": [230, 234, 789, 105, 318], "5733d4c8d058e614000b6354": [230, 234, 789, 318, 103], "5733d4c8d058e614000b6355": [230, 234, 51, 209, 162], "5733d4c8d058e614000b6356": [230, 234, 318, 232, 14], "5733d68ed058e614000b637f": [231, 234, 49, 224, 10], "5733d68ed058e614000b6380": [231, 49, 51, 53, 161], "5733d68ed058e614000b6381": [10, 49, 789, 51, 7], "5733f1784776f41900661575": [232, 233, 162, 11, 110], "5733f1784776f41900661576": [232, 113, 115, 162, 49], "5733f1784776f41900661577": [162, 115, 91, 110, 111], "5733f309d058e614000b6648": [233, 232, 51, 231, 162], "5733f309d058e614000b6649": [54, 223, 50, 126, 233], "5733f309d058e614000b664a": [233, 220, 318, 228, 126], "5733fb7bd058e614000b66ff": [234, 233, 232, 70, 11], "5733fb7bd058e614000b6700": [234, 224, 52, 162, 14], "5733fb7bd058e614000b6701": [234, 230, 162, 50, 231], "57373d0cc3c5551400e51e85": [217, 235, 236, 52, 805], "57373d0cc3c5551400e51e86": [235, 236, 217, 238, 542], "57373d0cc3c5551400e51e87": [235, 116, 60, 767, 441], "57373d0cc3c5551400e51e88": [235, 297, 109, 84, 60], "57373d0cc3c5551400e51e89": [235, 236, 237, 717, 556], "5737821cc3c5551400e51f19": [236, 237, 179, 70, 235], "5737821cc3c5551400e51f1a": [236, 367, 61, 110, 197], "5737821cc3c5551400e51f1b": [236, 235, 699, 294, 882], "5737821cc3c5551400e51f1c": [236, 235, 194, 73, 69], "57378b141c4567190057449f": [237, 236, 235, 179, 674], "57378b141c456719005744a0": [237, 235, 105, 72, 238], "57378b141c456719005744a1": [237, 238, 235, 775, 23], "57378b141c456719005744a2": [237, 882, 105, 171, 699], "57378c9b1c456719005744a8": [238, 237, 778, 254, 476], "57378c9b1c456719005744a9": [238, 236, 775, 237, 661], "57378c9b1c456719005744a7": [238, 297, 237, 25, 699], "57378c9b1c456719005744aa": [238, 236, 690, 882, 876], "5737a25ac3c5551400e51f51": [239, 424, 443, 339, 137], "5737a25ac3c5551400e51f52": [239, 204, 203, 237, 107], "5737a25ac3c5551400e51f53": [236, 238, 239, 628, 820], "5737a25ac3c5551400e51f54": [239, 238, 203, 204, 490]}}'
JOBS = {'0': [{'kind': 'panel', 'tag': 'panel-qwen3-0.6b-base', 'model': 'Qwen/Qwen3-0.6B-Base', 'qa': True}, {'kind': 'cpt', 'rows_per_step': 8, 'micro': 4, 'eval_every': 50, 'qa': True, 'tag': 'cpt-r0-lr3e-5', 'replay': 0.0, 'peak_lr': 3e-05, 'steps': 320, 'interp': [0.25, 0.5, 0.75]}, {'kind': 'cpt', 'rows_per_step': 8, 'micro': 4, 'eval_every': 50, 'qa': True, 'tag': 'cpt-r5-lr3e-5', 'replay': 0.05, 'peak_lr': 3e-05, 'steps': 320, 'interp': [0.25, 0.5, 0.75], 'save': True}, {'kind': 'qlora', 'tag': 'qlora-cpt-r5-lr3e-5', 'model': '/kaggle/working/ckpt_cpt-r5-lr3e-5', 'epochs': 2, 'batch': 8, 'lr': 0.0002}, {'kind': 'qlora', 'tag': 'qlora-qwen3-0.6b-base', 'model': 'Qwen/Qwen3-0.6B-Base', 'epochs': 2, 'batch': 8, 'lr': 0.0002}], '1': [{'kind': 'panel', 'tag': 'panel-qwen3.5-0.8b-base', 'model': 'Qwen/Qwen3.5-0.8B-Base', 'qa': True}, {'kind': 'cpt', 'rows_per_step': 8, 'micro': 4, 'eval_every': 50, 'qa': True, 'tag': 'cpt-r25-lr3e-5', 'replay': 0.25, 'peak_lr': 3e-05, 'steps': 320, 'interp': [0.25, 0.5, 0.75]}, {'kind': 'cpt', 'rows_per_step': 8, 'micro': 4, 'eval_every': 50, 'qa': True, 'tag': 'cpt-r5-lr1e-4', 'replay': 0.05, 'peak_lr': 0.0001, 'steps': 320, 'interp': [0.5]}, {'kind': 'extinit', 'tag': 'extinit-default', 'new_tokens': 10000, 'mark_regex': False}, {'kind': 'cpt', 'rows_per_step': 8, 'micro': 2, 'eval_every': 50, 'qa': True, 'tag': 'cpt-ext-mean-r5-lr3e-5', 'replay': 0.05, 'peak_lr': 3e-05, 'steps': 320, 'interp': [0.5], 'init': 'mean', 'tokenizer': True, 'new_tokens': 10000, 'mark_regex': False}]}

import json, os, subprocess, sys, time, platform, itertools
t_start = time.time()
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==5.22.1", "peft==0.21.2", "bitsandbytes==0.50.2",
                "accelerate==1.15.0", "datasets==5.1.0", "tokenizers==0.23.3", "huggingface_hub==2.4.0"], check=False)
W = "/kaggle/working"
os.makedirs(W + "/data", exist_ok=True)
for name, text in FILES.items():
    open(W + "/" + name, "w", encoding="utf-8").write(text)
open(W + "/data/qa_split.json", "w", encoding="utf-8").write(SPLIT)
from datasets import load_dataset
def dump(name, ds, n):
    import hashlib
    h = hashlib.sha256(); k = 0
    with open(W + "/data/" + name + ".jsonl", "w", encoding="utf-8") as f:
        for r in itertools.islice(ds, n):
            f.write(json.dumps({"id": r["id"], "text": r["text"]}, ensure_ascii=False) + "\n"); h.update(r["id"].encode()); k += 1
    print(name, k, h.hexdigest()[:16], flush=True)
dump("hi_test", load_dataset("HuggingFaceFW/fineweb-2", name="hin_Deva", split="test", streaming=True), 1000)
dump("en_edu", load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True), 8000)
dump("hi_train", load_dataset("HuggingFaceFW/fineweb-2", name="hin_Deva", split="train", streaming=True), 20000)
print("setup min", round((time.time() - t_start) / 60, 1), flush=True)


In [ ]:

import glob
CKPT_DIR = glob.glob("/kaggle/input/**/ckpt_cpt-r5-lr1e-4", recursive=True)
for g, lst in JOBS.items():
    for j in lst:
        if str(j.get("model", "")).startswith("CKPT:"):
            j["model"] = CKPT_DIR[0] if CKPT_DIR else "MISSING"
procs = {}
for g, lst in JOBS.items():
    json.dump(lst, open(W + "/jobs_" + g + ".json", "w"))
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=g)
    procs[g] = subprocess.Popen([sys.executable, W + "/worker_da.py", W + "/jobs_" + g + ".json", W],
                                stdout=open(W + "/log_gpu" + g + ".txt", "w"), stderr=subprocess.STDOUT, env=env)
while any(p.poll() is None for p in procs.values()):
    time.sleep(60)
    print(round((time.time() - t_start) / 60, 1), "min", {g: open(W + "/log_gpu" + g + ".txt").read()[-200:].replace("\n", " | ") for g in procs}, flush=True)
ALL = {}
for f in sorted(glob.glob(W + "/res_*.json")):
    r = json.load(open(f)); ALL[r["tag"]] = {"ok": r.get("ok"), "wallSec": r.get("wallSec"), "error": r.get("error")}
ALL["totalMin"] = round((time.time() - t_start) / 60, 1)
json.dump(ALL, open(W + "/summary.json", "w"), indent=1)
print(json.dumps(ALL, indent=1))
